# population

In [ ]:
#| default_exp game/flag

In [ ]:
#| export
from fastcore.basics import patch
import inspect
import copy
import colorsys
from importlib import resources

# fun with colors
import colorsys
import seaborn as sns
import matplotlib.pyplot as plt
from importlib import resources
import pandas as pd
import random
import re
from enum import Enum

In [ ]:
#| export
from HexMagic.styles import   SVGBuilder, SVGDef,  Generatable, NamedColor, StyleCSS
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord , Hex, HexGrid, PrimitiveDemo, HexWrapper, HexTouchMap
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion ,  unique_windy_edge
from HexMagic.terrainpatterns import TerrainPatterns, PathPattern
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.terrain import Terrain


In [ ]:
#| export
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, OverlaySpec
from HexMagic.overlay import OverlayContext

import chess.svg

In [ ]:
#| export
class PieceType(Enum):
    """Chess piece types."""
    KING   = "king"
    QUEEN  = "queen"
    BISHOP = "bishop"
    KNIGHT = "knight"
    ROOK   = "rook"
    PAWN   = "pawn"

    @property
    def icon(self):
        return {"king": "♚", "queen": "♛", "bishop": "♝",
                "knight": "♞", "rook": "♜", "pawn": "♟"}[self.value]


In [ ]:
#| export
class CountryFlag:

    def __init__(self, c, name, year, patternIndex=0):
        h, s, v = colorsys.rgb_to_hsv(*c)
        
        self.primary = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb(h, s * 0.4, min(v * 1.4, 1.0)))
        self.darkPrimary = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb(h, s, max(v * 0.35, 0.05)))
        self.lightPrimary = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb(h, s * 0.12, min(v * 1.4, 0.97)))
        
        self.comp = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb((h + 0.5) % 1, s, max(v * 0.35, 0.05)))
        self.lightComp = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb((h + 0.5) % 1, s * 0.15, min(v * 1.3, 0.95)))

        self.basePrimary = colorsys.rgb_to_hsv(*c)
        self.baseComp = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb((h + 0.5) % 1, s, v))
        self.tri1 = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb((h + 1/3) % 1, s, v))
        self.tri2 = plt.matplotlib.colors.rgb2hex(
            colorsys.hsv_to_rgb((h - 1/3) % 1, s, v))
        
        self.name = name
        self.year = year
        self.countryPrefix = CountryFlag.countryName(name)
        self.capital = CountryFlag.cityName(name)
        self.patternIndex = patternIndex

    @classmethod
    def seaborn(cls,name:str,levels=7,gender=None,year=1900):
        palette = sns.color_palette(name, levels)

        if gender is not None:
            names = random.sample(CountryFlag.commonNames(year=year,gender=gender),levels)
        else:
            allN = CountryFlag.commonNames(year=year,gender="M") + CountryFlag.commonNames(year=year,gender="F")
            names = random.sample(allN,levels)
            
       
        ret = []
        for i, color in enumerate(palette):
            ret.append(cls(color,name=names[i],year=year))
        return ret

    
    @classmethod
    def commonNames(cls,year=1900,gender="M"):
        
        with resources.files('HexMagic').joinpath('data/misc/popularNames.csv').open() as f:
            df = pd.read_csv(f)
        
        filtered = df[(df['Year'] == year) & (df['Gender'] == gender)]
        if filtered.empty:
            raise ValueError(f"No names found for year={year}, gender={gender}")
        
        # Weight by count for more realistic distribution
        return filtered['Name'].tolist()


    @classmethod
    def countryName(cls, name , pattern=None, descriptor=None):
        # Dictionary of place descriptors organized by first letter
        PLACE_DESCRIPTORS = {
            'A': ['Abbey', 'Acres', 'Alcove', 'Apex', 'Archipelago', 'Arena', 'Atoll', 'Avenue'],
            'B': ['Basin', 'Bay', 'Bluff', 'Borough', 'Boundary', 'Bower', 'Burg', 'Borderlands'],
            'C': ['Canyon', 'Cape', 'Castle', 'Citadel', 'Clearing', 'Cove', 'Crossing', 'County'],
            'D': ['Dale', 'Dell', 'Delta', 'Den', 'District', 'Domain', 'Dunes', 'Dominion'],
            'E': ['Edge', 'Enclave', 'End', 'Estate', 'Expanse', 'Empire', 'Escarpment', 'Eyrie'],
            'F': ['Falls', 'Fen', 'Fjord', 'Forest', 'Fort', 'Frontier', 'Fields', 'Fief'],
            'G': ['Gap', 'Garden', 'Gate', 'Glade', 'Glen', 'Gorge', 'Grotto', 'Grove'],
            'H': ['Habitat', 'Harbor', 'Haven', 'Heath', 'Heights', 'Hideaway', 'Hill', 'Hollow'],
            'I': ['Isle', 'Inlet', 'Island', 'Isthmus', 'Ironworks', 'Imperium', 'Inn', 'Impasse'],
            'J': ['Junction', 'Jungle', 'Jetty', 'Juncture', 'Jurisdiction', 'Jut', 'Joint', 'Jewel'],
            'K': ['Keep', 'Kingdom', 'Knoll', 'Key', 'Knot', 'Kiosk', 'Krantz', 'Karst'],
            'L': ['Lagoon', 'Lake', 'Landing', 'Land', 'Lair', 'Ledge', 'Lodge', 'Lowlands'],
            'M': ['Manor', 'Marsh', 'Meadow', 'Mesa', 'Moor', 'Mount', 'Mountains', 'Mound'],
            'N': ['Narrows', 'Nest', 'Niche', 'Nook', 'North', 'Notch', 'Nation', 'Neighborhood'],
            'O': ['Oasis', 'Observatory', 'Outpost', 'Overlook', 'Orchard', 'Outcrop', 'Outlet', 'Outlands'],
            'P': ['Palace', 'Park', 'Pass', 'Path', 'Peak', 'Peninsula', 'Pinnacle', 'Plaza', 'Point', 'Province'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Quarters', 'Quad', 'Quadrant', 'Quest', 'Quietude'],
            'R': ['Range', 'Ravine', 'Reach', 'Realm', 'Reef', 'Region', 'Reserve', 'Retreat', 'Ridge', 'Rise'],
            'S': ['Sanctuary', 'Settlement', 'Shire', 'Shore', 'Slopes', 'Sound', 'Span', 'Spring', 'Summit', 'Stronghold'],
            'T': ['Terrace', 'Territory', 'Thicket', 'Timberland', 'Tower', 'Town', 'Trail', 'Trench', 'Tundra', 'Township'],
            'U': ['Undergrowth', 'Underpass', 'Union', 'Uplands', 'Upper', 'Utopia', 'Utterness', 'Umbrage'],
            'V': ['Vale', 'Valley', 'Vault', 'View', 'Villa', 'Village', 'Vineyards', 'Vista', 'Void', 'Vanguard'],
            'W': ['Ward', 'Wasteland', 'Water', 'Way', 'Wetlands', 'Wilds', 'Wood', 'Woods', 'Works', 'Warren'],
            'X': ['Xanadu', 'Xenolith', 'Xerophyte', 'X-Roads', 'Xeric', 'Xyst', 'X-Point', 'X-ing'],
            'Y': ['Yard', 'Yonder', 'Yurt', 'Yards', 'Yielding', 'York', 'Yukon', 'Yews'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zigzag', 'Ziggurat', 'Zion', 'Zodiac', 'Zocalo'],
        }

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = PLACE_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Precinct'])
        
        # Choose descriptor
        if descriptor and descriptor in descriptors:
            place = descriptor
        else:
            place = random.choice(descriptors)

        return place
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)

    @classmethod
    def cityName(cls, name , pattern=None, descriptor=None, use_suffix=None):
        # Dictionary of place descriptors organized by first letter
        SETTLEMENT_DESCRIPTORS = {
            'A': ['Acres', 'Arbor', 'Ashton', 'Auburn', 'Avon', 'Aldridge', 'Ashford', 'Aston'],
            'B': ['Bay', 'Beach', 'Bridge', 'Brook', 'Burg', 'Borough', 'Bluff', 'Bend'],
            'C': ['City', 'Cove', 'Creek', 'Crest', 'Crossing', 'Center', 'Cape', 'Corners'],
            'D': ['Dale', 'Dell', 'Dunes', 'Down', 'Dock', 'Delta', 'Downs', 'Den'],
            'E': ['End', 'Edge', 'Estates', 'Elms', 'Enclave', 'Evergreen', 'East', 'Elm'],
            'F': ['Falls', 'Field', 'Fields', 'Ford', 'Forest', 'Fort', 'Forks', 'Ferry'],
            'G': ['Glen', 'Glade', 'Green', 'Grove', 'Gate', 'Gardens', 'Groves', 'Gap'],
            'H': ['Harbor', 'Haven', 'Heights', 'Hill', 'Hills', 'Hollow', 'Heath', 'Hurst'],
            'I': ['Isle', 'Island', 'Inlet', 'Inn', 'Ironworks', 'Ivy', 'Isles', 'Inches'],
            'J': ['Junction', 'Jetty', 'Juncture', 'Junction', 'Jamestown', 'Jardin', 'Jct', 'Joya'],
            'K': ['Key', 'Knoll', 'Knolls', 'Keep', 'Keystone', 'Kingswood', 'Kirk', 'Knolle'],
            'L': ['Lake', 'Landing', 'Lawn', 'Ledge', 'Lock', 'Lodge', 'Lagoon', 'Lynn'],
            'M': ['Manor', 'Meadow', 'Meadows', 'Mill', 'Mills', 'Mount', 'Moor', 'Mountain'],
            'N': ['North', 'Nook', 'Narrows', 'Neck', 'Nest', 'Newton', 'New', 'Notch'],
            'O': ['Oaks', 'Orchard', 'Overlook', 'Outpost', 'Outlet', 'Oak', 'Oasis', 'Old'],
            'P': ['Park', 'Pines', 'Plains', 'Point', 'Pond', 'Port', 'Plaza', 'Pass'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Queen', 'Quarters', 'Quayside', 'Quest', 'Quince'],
            'R': ['Ridge', 'River', 'Rock', 'Run', 'Ranch', 'Rapids', 'Reach', 'Rest'],
            'S': ['Springs', 'Shore', 'Shores', 'South', 'Station', 'Summit', 'Shire', 'Side'],
            'T': ['Town', 'Terrace', 'Trace', 'Trail', 'Township', 'Tower', 'Thicket', 'Timber'],
            'U': ['Union', 'Uplands', 'Upper', 'Underwood', 'Unity', 'University', 'Upton', 'Utopia'],
            'V': ['Vale', 'Valley', 'View', 'Villa', 'Village', 'Vista', 'Ville', 'Vineyards'],
            'W': ['West', 'Water', 'Waters', 'Way', 'Wells', 'Wood', 'Woods', 'Wick'],
            'X': ['Xanadu', 'X-Roads', 'Xing', 'Xavier', 'Xeric', 'Xenia', 'Xenophon', 'Xyst'],
            'Y': ['Yard', 'Yonder', 'York', 'Yards', 'Yew', 'Yews', 'Yale', 'Yarmouth'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zion', 'Zinc', 'Zodiac', 'Zona', 'Zuni'],
        }

        # Naming patterns for settlements
        PATTERNS = [
            "{name}ville",           # Karlville
            "{name}ton",             # Karlton
            "{name}burg",            # Karlburg
            "{name}wood",            # Karlwood
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
            "New {name}",            # New Karl
            "Old {name}",            # Old Karl
            "Little {name}",         # Little Karl
            "Upper {name}",          # Upper Karl
            "Lower {name}",          # Lower Karl
            "East {name}",           # East Karl
            "West {name}",           # West Karl
            "North {name}",          # North Karl
            "South {name}",          # South Karl
        ]

        # Shorter patterns for descriptors (avoid double suffixes)
        DESCRIPTOR_PATTERNS = [
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = SETTLEMENT_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Plaza'])
        
        # Decide whether to use suffix or descriptor pattern
        if use_suffix is None:
            use_suffix = random.choice([True, False])
        
        if use_suffix:
            # Use simple suffix patterns (first 7 patterns)
            patterns = PATTERNS[:7]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            
            if "{place}" in template:
                place = descriptor if descriptor and descriptor in descriptors else random.choice(descriptors)
                return template.format(name=name, place=place)
            else:
                return template.format(name=name)
        else:
            # Use directional/size prefix patterns (last 9 patterns)
            patterns = PATTERNS[7:]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            return template.format(name=name)

    @staticmethod
    def _render_attrs(attrs: dict = None) -> str:
        """Render a dict of HTML/HTMX attributes into an SVG attribute string."""
        if not attrs:
            return ""
        return " " + " ".join(f"{k}='{v}'" for k, v in attrs.items())

    @staticmethod
    def decode(s: str) -> 'CountryFlag':
        """Decode CountryFlag from string."""
        parts = s.split('|')
        primary = parts[0]
        name = parts[1]
        year = int(parts[2])
        if len(parts) > 3:
            patternIndex = int(parts[3])
        else:
            patternIndex = 0
        
        # Convert hex back to RGB tuple (0-1 range)
        rgb = plt.matplotlib.colors.to_rgb(primary)
        
        return CountryFlag(rgb, name=name, year=year,patternIndex=patternIndex)

    @staticmethod
    def test_flag() -> 'CountryFlag':
        """Return a CountryFlag for testing with name 'Merlin' and purple color."""
        purple = colorsys.hsv_to_rgb(0.75, 0.7, 0.8)  # a nice purple
        return CountryFlag(purple, name="Merlin", year=1900)


I need a staticmethod that would return a country flag for testing. Can you make the name be Merlin with a purple color?

In [ ]:
#| export
@patch
def encode(self: CountryFlag) -> str:
    """Encode CountryFlag to a single line string."""
    # Format: primary|name|year
    return f"{self.primary}|{self.name}|{self.year}|{self.patternIndex}"

In [ ]:
#| export
@patch
def plain(self:CountryFlag,name,width=3):
    return StyleCSS(name,fill=self.primary,stroke=self.comp,stroke_width=width)

@patch
def kingStyle(self:CountryFlag,name,width=2):
    #saturation = 0.7
    style = self.plain(name,width)
    style.name = f"country_{name}"
    #style.properties["fill"] = style.desaturate(saturation).properties["fill"]
    style.properties["opacity"] = 0.7
    return style

@patch
def contrastStyle(self:CountryFlag,name,width=1.5):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.comp,
            stroke="#000",
            stroke_width=width
        )

@patch
def labelStyle(self:CountryFlag, name, width=0):
    return StyleCSS(
        f"contrast_{name}",
        fill=self.darkPrimary,
        stroke="none",
        stroke_width=width
    )


I want to thing in terms of color blindness for our colors. I think primary should be more of our background color and should be slightly lighter and comp could be for text and slightly darker and could be used for labels

Is this at that ratio?

In [ ]:
flags = CountryFlag.seaborn("husl",4)
for flag in flags:
    print(flag.name , flag.capital)


Edward Edge of Edward
Florence Florencewood
Joseph North Joseph
Bertha Old Bertha


In [ ]:
hexCount = 9
radius = 30
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    style = flat.plain(f"flag_{i}",width=10)
    sampleHex = Hex(radius=30, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_zCJ-d3pSRqyZ-InWmasMGA&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;630&quot; height=&quot;80&quot; viewBox=&quot;0 0 630 80&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;style&gt;
.flag_8 {
  fill:#d9ffd9;
  stroke:#462c46;
  stroke-width:10;
}
.flag_7 {
  fill:#8f8f8f;
  stroke:#242424;
  stroke-width:10;
}
.flag_6 {
  fill:#ffcaa5;
  stroke:#082b43;
  stroke-width:10;
}
.flag_5 {
  fill:#ff9acf;
  stroke:#015428;
  stroke-width:10;
}
.flag_4 {
  fill:#b3d0f6;
  stroke:#3e2b14;
  stroke-width:10;
}
.flag_3 {
  fill:#ffffd6;
  stroke:#363659;
  stroke-width:10;
}
.flag_2 {
  fill:#ffe6cf;
  stroke:#2f4459;
  stroke-width:10;
}
.flag_1 {
  fill:#f4edff;
  stroke:#454a3d;
  stroke-width:10;
}
.flag_0 {
  fill:#d9ffd9;
  stroke:#462c46;
  stroke-width:10;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;hex-0&quot;&gt;
&lt;polygon points=&quot;65,25 65,55 40,70 14,55 14,24 39,10 &quot; class=&quot;flag_0&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-1&quot;&gt;
&lt;polygon points=&quot;135,25 135,55 110,70 84,55 84,24 110,10 &quot; class=&quot;flag_1&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-2&quot;&gt;
&lt;polygon points=&quot;205,25 205,55 180,70 154,55 154,24 180,10 &quot; class=&quot;flag_2&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-3&quot;&gt;
&lt;polygon points=&quot;275,25 275,55 250,70 224,55 224,24 250,10 &quot; class=&quot;flag_3&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-4&quot;&gt;
&lt;polygon points=&quot;345,25 345,55 320,70 294,55 294,24 320,10 &quot; class=&quot;flag_4&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-5&quot;&gt;
&lt;polygon points=&quot;415,25 415,55 390,70 364,55 364,24 390,10 &quot; class=&quot;flag_5&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-6&quot;&gt;
&lt;polygon points=&quot;485,25 485,55 460,70 434,55 434,24 460,10 &quot; class=&quot;flag_6&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-7&quot;&gt;
&lt;polygon points=&quot;555,25 555,55 530,70 504,55 504,24 530,10 &quot; class=&quot;flag_7&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;hex-8&quot;&gt;
&lt;polygon points=&quot;625,25 6

In [ ]:
#| export


@patch
def circlePattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'<rect width=120 height= 90 fill="{self.comp}"/><circle cx="{50}" cy="{45}" r="{30}" fill="{self.primary}"/>'
        return SVGDef("pattern", id, content, 
                    width=120, height=90, 
                    patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def triPattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'''<rect width=96 height= 96 fill="{self.primary}"/>
        <circle cx="{48}" cy="{48}" r="{48}" fill="{self.comp}"/>
        <circle cx="{48}" cy="{48}" r="{32}" fill="{self.tri1}"/>
        <circle cx="{48}" cy="{48}" r="{16}" fill="{self.tri2}"/>'''
        
        return SVGDef("pattern", id, content, 
                    width=96, height=96, 
                    patternUnits="userSpaceOnUse")

In [ ]:
#| export
@patch
def swirl(self:CountryFlag,id):

    content = f"""<g  fill='{self.primary}'><rect width=400 height=400 fill='{self.primary}' /></g>
<g  fill='{self.comp}' fill-opacity='1'><path d='M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z'/><circle  cx='200' cy='1' r='60'/><circle  cx='200' cy='400' r='60'/></g><g  fill='{self.primary}'><circle  cx='0' cy='200' r='60'/><circle  cx='400' cy='200' r='60'/></g>
"""
    pat = SVGDef("pattern", id, content,
                  width=400, height=400,
                  patternUnits="userSpaceOnUse")
    scale = 0.2
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.swirl(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_0XPjPoicSQqtmYr3Ix___w&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1890&quot; height=&quot;220&quot; viewBox=&quot;0 0 1890 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot; patternTransform=&quot;scale(0.2)&quot;&gt;&lt;g  fill=&#x27;#d9ffd9&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#d9ffd9&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#462c46&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#d9ffd9&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;
&lt;/pattern&gt;
&lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_1_pat&quot; patternTransform=&quot;scale(0.2)&quot;&gt;&lt;g  fill=&#x27;#f4edff&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#f4edff&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#454a3d&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  

In [ ]:
#| export
@patch
def yin(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary}' width='800' height='800'/>
     	<g id="YinYang" fill="{self.comp}" stroke="none" stroke-width="0" fill-rule="evenodd">
		<title>Yin-Yang, by Adam Stanislav</title>
		<desc>The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .</desc>

		<!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. -->
		<path d="M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z" stroke='black' stroke-width='25' stroke-opacity='1' stroke-linecap='square' />
	</g>
  """


    pat = SVGDef("pattern", id, content,
                  width=800, height=800,
                  patternUnits="userSpaceOnUse")
    scale = 0.1 
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_mARnL1K8QT2x_xbA1VWQ_g&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1890&quot; height=&quot;220&quot; viewBox=&quot;0 0 1890 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;800&quot; height=&quot;800&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect fill=&#x27;#d9ffd9&#x27; width=&#x27;800&#x27; height=&#x27;800&#x27;/&gt;
     	&lt;g id=&quot;YinYang&quot; fill=&quot;#462c46&quot; stroke=&quot;none&quot; stroke-width=&quot;0&quot; fill-rule=&quot;evenodd&quot;&gt;
		&lt;title&gt;Yin-Yang, by Adam Stanislav&lt;/title&gt;
		&lt;desc&gt;The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .&lt;/desc&gt;

		&lt;!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. --&gt;
		&lt;path d=&quot;M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z&quot; stroke=&#x27;black&#x27; stroke-width=&#x27;25&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#

How do I fill black in yin

In [ ]:
#| export
@patch
def weave(self:CountryFlag, id):
    
    content = f"""<rect fill='{self.primary}' width='600' height='600'/><path  fill='none' stroke='{self.comp}' stroke-width='55' stroke-opacity='1' stroke-linecap='square' d='M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200'/>
    """
    pat = SVGDef("pattern", id, content,
                  width=600, height=600,
                  patternUnits="userSpaceOnUse")
    scale = 0.25
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
hexCount = 19
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("magma", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.weave(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_70DmVKa0RriyiTBgJniqHw&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;3990&quot; height=&quot;220&quot; viewBox=&quot;0 0 3990 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;600&quot; height=&quot;600&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot; patternTransform=&quot;scale(0.25)&quot;&gt;&lt;rect fill=&#x27;#191924&#x27; width=&#x27;600&#x27; height=&#x27;600&#x27;/&gt;&lt;path  fill=&#x27;none&#x27; stroke=&#x27;#0c0d03&#x27; stroke-width=&#x27;55&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#x27;square&#x27; d=&#x27;M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200&#x27;/&gt;
    &lt;/pattern&gt;
&lt;pattern width=&quot;600&quot; height=&quot;600&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_1_pat&quot; patternTransform=&quot;scale(0.25)&quot;&gt;&lt;rect fill=&#x27;#39354c&#x27; width=&#x27;600&#x27; height=&#x27;600&#x27;/&gt;&lt;path  fill=&#x27;none&#x27; stroke=&#x27;#111305&#x27; stroke-width=&#x27;55&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#x27;square&#x27; d=&#x27;M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200&#x27;/&gt;
    &lt;/pattern&gt;
&lt;pattern width=&quot;600&quot; height=&quot;600&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_2_pat&quot; patternTransform=&quot;scale(0.25)&quot;&gt;&lt;rect fill=&#x27;#5d5177&#x27; width=&#x27;600&#x27; height=&#x27;600&#x27;/&gt;&lt;path  fill=&#x27;none&#x27; stroke=&#x27;#171e06&#x27; stroke-width=&#x27;55&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#x27;square&#x27; d=&#x27;M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200&#x27;/&gt;
    &lt;/patt

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_3BoRFI9AQhaa8tyWDUZ09A&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1890&quot; height=&quot;220&quot; viewBox=&quot;0 0 1890 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;800&quot; height=&quot;800&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect fill=&#x27;#d9ffd9&#x27; width=&#x27;800&#x27; height=&#x27;800&#x27;/&gt;
     	&lt;g id=&quot;YinYang&quot; fill=&quot;#462c46&quot; stroke=&quot;none&quot; stroke-width=&quot;0&quot; fill-rule=&quot;evenodd&quot;&gt;
		&lt;title&gt;Yin-Yang, by Adam Stanislav&lt;/title&gt;
		&lt;desc&gt;The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .&lt;/desc&gt;

		&lt;!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. --&gt;
		&lt;path d=&quot;M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z&quot; stroke=&#x27;black&#x27; stroke-width=&#x27;25&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.triPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_G1koc2biSLm9_9y17ODiMQ&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1890&quot; height=&quot;220&quot; viewBox=&quot;0 0 1890 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#d9ffd9&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#462c46&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#7f7fc9&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#c97f7f&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_1_pat&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#f4edff&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#454a3d&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#d4beae&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#aed4be&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_2_pat&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#ffe6cf&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#2f4459&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#86fdc0&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#c086fd&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_3_pat&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#ffffd6&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#363659&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#99ffff&quot;/&gt;
        &lt;

In [ ]:
for pat in canvas.definitions:
    print(pat.attributes["id"])

CountryStar_0_pat
CountryStar_1_pat
CountryStar_2_pat
CountryStar_3_pat
CountryStar_4_pat
CountryStar_5_pat
CountryStar_6_pat
CountryStar_7_pat
CountryStar_8_pat


In [ ]:


hexStyles = CountryFlag.seaborn("Accent", hexCount)
hexIndex = 3
name = f"CountrySwirl_{hexIndex}"
patternName = f"{name}_pat"
hexPat = hexStyles[hexIndex].swirl(patternName)

#some drawing setup 
hexStyle = StyleCSS(name, fill=f"url(#{patternName})")
canvas = SVGBuilder()
canvas.width=200 ;canvas.height=200
canvas.add_style(hexStyle)
canvas.add_definition(hexPat)

# add our hex to the canvas
sampleHex = Hex(radius=50,center=MapCord(100,100),style=hexStyle)
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()



HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_yaxVQMg9TLuoJ-8SjH3gAg&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;200&quot; height=&quot;200&quot; viewBox=&quot;0 0 200 200&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountrySwirl_3_pat&quot; patternTransform=&quot;scale(0.2)&quot;&gt;&lt;g  fill=&#x27;#ffffd6&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#ffffd6&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#363659&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#ffffd6&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;
&lt;/pattern&gt;
  &lt;/defs&gt;
  &lt;style&gt;
.CountrySwirl_3 {
  fill:url(#CountrySwirl_3_pat);
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;main&quot;&gt;
&lt;polygon points=&quot;143,75 143,125 100,150 56,125 56,75 99,50 &quot; class=&quot;CountrySwirl_3&quot;/&gt;
&lt;/g&gt;
&lt;/svg&gt;&lt;/div&gt;
  &lt;/body&gt;
&lt;/html&gt;
" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; ful

In [ ]:
#| export
@patch
def sheridan(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary} width='1600' height='900'/><path  fill='{self.tri1}' d='M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z'/><path  fill='{self.tri2}' d='M751.6 450.5c0-49.5 99.8-130.3 99.8-180c0-49.7-99.8-130.5-99.8-180s49.4-90 49.4-90v-0.5h-802v900h793c0 0-40.4-43-40.4-89.5c0-49.5 99.8-130.3 99.8-180s-99.8-130.5-99.8-180z'/>"""
    pat = SVGDef("pattern", id, content,
                  width=1600, height=900,
                  patternUnits="userSpaceOnUse")
    scale = 0.1
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

Lets redo sheridan so it is vertical waves of comp and primary

It does look good as a repeated pattern. Can you build a better version?

Can you make chess pieces svg for our country?

use the built-in chess.svg piece shapes (recolored with CountryFlag colors),
all 6.
we probaly need some sort of scale and center coordinates to place on a map

It might be easier to just take the raw svg and build our own defs and not use the libary (other than for building our functions

piece_svgs = {}
for sym in ["K", "Q", "B", "N", "R", "P"]:
    raw = chess.svg.piece(chess.Piece.from_symbol(sym))
    # Extract just the inner <g> content
    inner = raw.split('>', 1)[1].rsplit('</svg>', 1)[0]
    piece_svgs[sym] = inner
    print(f"--- {sym} ---")
    print(inner, "\n")


## Piece of mind

In [ ]:
#| export
@patch
def flag_hex(flag:CountryFlag, radius=25):
    """Hex with flag pattern fill, built via SVGBuilder."""
    pad = 4
    size = radius * 2 + pad * 2
    center = MapCord(size / 2, size / 2)

    builder = SVGBuilder()
    builder.width = size
    builder.height = size

    # Register flag pattern as a fill
    pat_name = f"fh_{flag.name}"
    builder.add_definition(flag.flagPattern(pat_name, scale=0.08))
    pat_style = StyleCSS(pat_name, fill=f"url(#{pat_name})",
                         stroke=flag.comp, stroke_width=3)
    builder.add_style(pat_style)

    h = Hex(radius=radius, center=center, style=pat_style)
    builder.adjust("hex", h.svg())

    return Div(NotStr(builder.xml()))

In [ ]:
!cat ../../docs/llms*

# HexMagic Computation & Database

> Data models, persistence, chunk management, spatial queries, and multi-scale terrain storage.

## Files Included
- `HexMagic/primitives.py` - Re-exports from plot modules (convenience imports)
- `HexMagic/database.py` - GeoStorage, DataProxy, chunk stitching, zoom pipeline

---

## HexMagic/primitives.py

```python
# Re-exports from plot modules (single source of truth)
from .plot.primitives import PrimitiveDemo, MapCord, MapSize, MapRect, MakeCord, MakeSize, MapPath
from .plot.hex import Hex, HexBackground, HexGrid, HexWrapper, LinearGradient, HexTouchMap, HexButtonGroup
from .plot.cube import HexPosition, GosperCurve
from .plot.region import HexRegion, windy_edge, variable_windy_edge, unique_windy_edge
from .plot.chunk import HexChunk
```

---

## HexMagic/database.py

```python
__all__ = ['LoadResult', 'SaveResult', 'ChunkRef', 'chunk_to_world', 'world_to_chunk', 'HexData', 'HexWeather', 'World', 'User',
           'ChunkBorder', 'ChunkMeta', 'Ge

Can you build something like the flag_hex but with a more traditional flag shape. Use our primitives as you can. we might want to be able to make this fasthtml/htmlx compatiable as well.

In [ ]:
#| export
@patch
def flag_banner(self: CountryFlag, width=150, height=100,
                pattern_fn=None, wavy=False, pole=True, attrs=None):
    """Traditional rectangular flag with optional pole and wavy edge.
    
    Returns SVGBuilder. Use .show() for Jupyter or wrap for FastHTML:
        Div(NotStr(flag.flag_banner().xml()), **htmx_attrs)
    
    Args:
        width/height: flag dimensions
        pattern_fn: 'swirl','weave','triPattern','circlePattern','yin','sheridan'
                    (defaults via patternIndex)
        wavy: wavy right edge via MapPath.make_windy
        pole: draw flagpole on left
        attrs: extra SVG attributes dict for the flag polygon
    """
    PATTERNS = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']
    if pattern_fn is None:
        pattern_fn = PATTERNS[self.patternIndex % len(PATTERNS)]
    
    pad, pole_w, finial_r, pole_extra = 6, 5, 6, 30
    
    # Layout offsets
    fx = pad + (pole_w if pole else 0)
    fy = pad + (finial_r * 2 if pole else 0)
    
    builder = SVGBuilder()
    builder.width = fx + width + pad
    builder.height = fy + height + (pole_extra if pole else 0) + pad
    
    # --- Pattern fill ---
    pat_id = f"fp_{self.name}"
    builder.add_definition(getattr(self, pattern_fn)(pat_id))
    flag_style = StyleCSS(f"flag_{self.name}",
                          fill=f"url(#{pat_id})", stroke=self.comp, stroke_width=2.5)
    builder.add_style(flag_style)
    
    # --- Flag shape via MapPath ---
    tl, tr = MapCord(fx, fy), MapCord(fx + width, fy)
    br, bl = MapCord(fx + width, fy + height), MapCord(fx, fy + height)
    
    if wavy:
        right_edge = MapPath([tr, br]).make_windy(iterations=3, offset_factor=0.12)
        points = [tl, tr] + right_edge.points[1:] + [bl]
    else:
        points = [tl, tr, br, bl]
    
    flag_path = MapPath(points, flag_style)
    builder.adjust("flag", flag_path.drawClosed(CountryFlag._render_attrs(attrs)))
    
    # --- Pole + finial ---
    if pole:
        pole_style = StyleCSS(f"pole_{self.name}",
                              stroke=self.darkPrimary, stroke_width=pole_w,
                              stroke_linecap="round", fill=self.darkPrimary)
        builder.add_style(pole_style)
        px = pad + pole_w / 2
        builder.adjust("pole",
            f'<line x1="{px}" y1="{pad + finial_r}" '
            f'x2="{px}" y2="{fy + height + pole_extra}" '
            f'class="{pole_style.name}"/>'
            f'<circle cx="{px}" cy="{pad + finial_r}" '
            f'r="{finial_r}" class="{pole_style.name}"/>')
    
    return builder


In [ ]:
#| export
@patch
def flag_html(self: CountryFlag, width=150, height=100,
              pattern_fn=None, wavy=False, pole=True, **htmx_attrs):
    """FastHTML-compatible flag. Pass any HTMX attrs as kwargs.
    
    Example: flag.flag_html(hx_get='/click', hx_target='#info')
    """
    builder = self.flag_banner(width=width, height=height,
                               pattern_fn=pattern_fn, wavy=wavy, pole=pole)
    return Div(NotStr(builder.xml()), **htmx_attrs)


In [ ]:
# Demo: show all pattern types
flags = CountryFlag.seaborn("husl", 6)
patterns = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']

canvas = SVGBuilder()
cols = len(patterns)
fw, fh = 150, 100
canvas.width = cols * (fw + 20) + 20
canvas.height = fh + 80

for i, (flag, pat) in enumerate(zip(flags, patterns)):
    sub = flag.flag_banner(width=fw, height=fh, pattern_fn=pat, pole=True, wavy=(i % 2 == 1))
    # Offset each flag horizontally
    x_off = 10 + i * (fw + 20)
    canvas.definitions.extend(sub.definitions)
    canvas.styles.update(sub.styles)
    wrapped = f'<g transform="translate({x_off}, 0)">'
    for layer in sub.layers:
        wrapped += layer.body
    wrapped += '</g>'
    canvas.adjust(f"flag_{i}", wrapped)

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_aztcof-qRveCFB9v9Tcx_g&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1040&quot; height=&quot;180&quot; viewBox=&quot;0 0 1040 180&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;fp_Willie&quot; patternTransform=&quot;scale(0.2)&quot;&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#ffc8d1&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#27564e&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;
&lt;/pattern&gt;
&lt;pattern width=&quot;600&quot; height=&quot;600&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;fp_Pearl&quot; patternTransform=&quot;scale(0.25)&quot;&gt;&lt;rect fill=&#x27;#ffecb4&#x27; width=&#x27;600&#x27; height=&#x27;600&#x27;/&gt;&lt;path  fill=&#x27;none&#x27; stroke=&#x27;#111e42&#x27; stroke-width=&#x27;55&#x27; stroke-opacity=&#x27;1&#x27; stroke-linecap=&#x27;square&#x27; d=&#x27;M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200&#x27;/&gt;
    &lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;fp_Gertrude&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#c1f7b0&quot;/&gt;
        &lt;circle

In [ ]:
#| export
@patch
def kingPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    """Place a chess king at center. attrs: dict of HTML/HTMX attributes for interactivity.
    For instance:
    Then usage looks like:
    
flag.kingPiece(center, scale=1.5, attrs={
    'hx-get': '/piece/king/clicked',
    'hx-target': '#game-panel',
    'hx-swap': 'innerHTML',
    'data-piece': 'king',
    'data-player': flag.name
})


    
    """
    if piece_id is None:
        piece_id = f"king_{self.name}"
    if fill is None:
        fill = self.primary
    
    paths = f'''<path d="M22.5 11.63V6M20 8h5" stroke-linejoin="miter" />
<path d="M22.5 25s4.5-7.5 3-10.5c0 0-1-2.5-3-2.5s-3 2.5-3 2.5c-1.5 3 3 10.5 3 10.5"
      fill="{fill}" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M11.5 37c5.5 3.5 15.5 3.5 21 0v-7s9-4.5 6-10.5c-4-6.5-13.5-3.5-16 4V27v-3.5c-3.5-7.5-13-10.5-16-4-3 6 5 10 5 10V37z"
      fill="{fill}" />
<path d="M11.5 30c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''


In [ ]:
#| export
@patch
def queenPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
               piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"queen_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M8 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM24.5 7.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM41 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM16 8.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM33 9a2 2 0 1 1-4 0 2 2 0 1 1 4 0z" />
<path d="M9 26c8.5-1.5 21-1.5 27 0l2-12-7 11V11l-5.5 13.5-3-15-3 15-5.5-14V25L7 14l2 12zM9 26c0 2 1.5 2 2.5 4 1 1.5 1 1 .5 3.5-1.5 1-1.5 2.5-1.5 2.5-1.5 1.5.5 2.5.5 2.5 6.5 1 16.5 1 23 0 0 0 1.5-1 0-2.5 0 0 .5-1.5-1-2.5-.5-2.5-.5-2 .5-3.5 1-2 2.5-2 2.5-4-8.5-1.5-18.5-1.5-27 0z" stroke-linecap="butt" />
<path d="M11.5 30c3.5-1 18.5-1 22 0M12 33.5c6-1 15-1 21 0" fill="none" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="{fill}" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def bishopPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
                piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"bishop_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<g fill="{fill}" stroke-linecap="butt">
<path d="M9 36c3.39-.97 10.11.43 13.5-2 3.39 2.43 10.11 1.03 13.5 2 0 0 1.65.54 3 2-.68.97-1.65.99-3 .5-3.39-.97-10.11.46-13.5-1-3.39 1.46-10.11.03-13.5 1-1.354.49-2.323.47-3-.5 1.354-1.94 3-2 3-2zM15 32c2.5 2.5 12.5 2.5 15 0 .5-1.5 0-2 0-2 0-2.5-2.5-4-2.5-4 5.5-1.5 6-11.5-5-15.5-11 4-10.5 14-5 15.5 0 0-2.5 1.5-2.5 4 0 0-.5.5 0 2zM25 8a2.5 2.5 0 1 1-5 0 2.5 2.5 0 1 1 5 0z" />
</g>
<path d="M17.5 26h10M15 30h15m-7.5-14.5v5M20 18h5" stroke-linejoin="miter" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def knightPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
                piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"knight_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M 22,10 C 32.5,11 38.5,18 38,39 L 15,39 C 15,30 25,32.5 23,18"
      fill="{fill}" stroke="{self.comp}" />
<path d="M 24,18 C 24.38,20.91 18.45,25.37 16,27 C 13,29 13.18,31.34 11,31 C 9.958,30.06 12.41,27.96 11,28 C 10,28 11.19,29.23 10,30 C 9,30 5.997,31 6,26 C 6,24 12,14 12,14 C 12,14 13.89,12.1 14,10.5 C 13.27,9.506 13.5,8.5 13.5,7.5 C 14.5,6.5 16.5,10 16.5,10 L 18.5,10 C 18.5,10 19.28,8.008 21,7 C 22,7 22,10 22,10"
      fill="{fill}" stroke="{self.comp}" />
<path d="M 9.5 25.5 A 0.5 0.5 0 1 1 8.5,25.5 A 0.5 0.5 0 1 1 9.5 25.5 z"
      fill="{self.comp}" stroke="{self.comp}" />
<path d="M 15 15.5 A 0.5 1.5 0 1 1 14,15.5 A 0.5 1.5 0 1 1 15 15.5 z"
      transform="matrix(0.866,0.5,-0.5,0.866,9.693,-5.173)"
      fill="{self.comp}" stroke="{self.comp}" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def rookPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"rook_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M9 39h27v-3H9v3zM12 36v-4h21v4H12zM11 14V9h4v2h5V9h5v2h5V9h4v5" stroke-linecap="butt" />
<path d="M34 14l-3 3H14l-3-3" />
<path d="M31 17v12.5H14V17" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M31 29.5l1.5 2.5h-20l1.5-2.5" />
<path d="M11 14h23" fill="none" stroke-linejoin="miter" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="{fill}" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def pawnPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"pawn_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M22.5 9c-2.21 0-4 1.79-4 4 0 .89.29 1.71.78 2.38C17.33 16.5 16 18.59 16 21c0 2.03.94 3.84 2.41 5.03-3 1.06-7.41 5.55-7.41 13.47h23c0-7.92-4.41-12.41-7.41-13.47 1.47-1.19 2.41-3 2.41-5.03 0-2.41-1.33-4.5-3.28-5.62.49-.67.78-1.49.78-2.38 0-2.21-1.79-4-4-4z"
      fill="{fill}" stroke="{self.comp}" stroke-width="1.5" stroke-linecap="round" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''


I want to add fasthtml style htmx clicking to these

Can you write the other methods? Did I patch King piece correctly

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
canvas = SVGBuilder()
canvas.width, canvas.height = 400, 80

pieces = [flag.kingPiece, flag.queenPiece, flag.bishopPiece, 
          flag.knightPiece, flag.rookPiece, flag.pawnPiece]

for i, fn in enumerate(pieces):
    canvas.adjust(f"p{i}", fn(MapCord(40 + i*60, 40), scale=1.2))

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_G6X9jRmaQS2wTm6t5nEOfQ&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;400&quot; height=&quot;80&quot; viewBox=&quot;0 0 400 80&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;p0&quot;&gt;
&lt;g id=&quot;king_Margaret&quot; fill=&quot;none&quot; fill-rule=&quot;evenodd&quot;
   stroke=&quot;#27564e&quot; stroke-width=&quot;1.5&quot;
   stroke-linecap=&quot;round&quot; stroke-linejoin=&quot;round&quot;
   transform=&quot;translate(40,40) scale(1.2) translate(-22.5,-22.5)&quot;
   style=&quot;cursor:pointer&quot;&gt;
&lt;path d=&quot;M22.5 11.63V6M20 8h5&quot; stroke-linejoin=&quot;miter&quot; /&gt;
&lt;path d=&quot;M22.5 25s4.5-7.5 3-10.5c0 0-1-2.5-3-2.5s-3 2.5-3 2.5c-1.5 3 3 10.5 3 10.5&quot;
      fill=&quot;#ffc8d1&quot; stroke-linecap=&quot;butt&quot; stroke-linejoin=&quot;miter&quot; /&gt;
&lt;path d=&quot;M11.5 37c5.5 3.5 15.5 3.5 21 0v-7s9-4.5 6-10.5c-4-6.5-13.5-3.5-16 4V27v-3.5c-3.5-7.5-13-10.5-16-4-3 6 5 10 5 10V37z&quot;
      fill=&quot;#ffc8d1&quot; /&gt;
&lt;path d=&quot;M11.5 30c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0&quot; /&gt;
&lt;/g&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;p1&quot;&gt;
&lt;g id=&quot;queen_Margaret&quot; fill=&quot;#ffc8d1&quot; fill-rule=&quot;evenodd&quot;
   stroke=&quot;#27564e&quot; stroke-width=&quot;1.5&quot;
   stroke-linecap=&quot;round&quot; stroke-linejoin=&quot;round&quot;
   transform=&quot;translate(100,40) scale(1.2) translate(-22.5,-22.5)&quot;
   style=&quot;cursor:pointer&quot;&gt;
&lt;path d=&quot;M8 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM24.5 7.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM41 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM16 8.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM33 9a2 2 0 1 1-4 0 2 2 0 1 1 4 0z&quot; /&gt;
&lt;path d=&quot;M9 26c8.5-1.5 21-1.5 27 0l2-12-7 11V11l-5.5 13.5-3-15-3 15-5.5-14V25L7 14l2 12zM9 26c0 2 1.5 2 2.5 4 1 1.5 1 1 .5 3.5-1.5 1-1.5 2.5-1.5 2.5-1.5 1.5.5 2.5.5 2.5 6.5 1 16.5 1 23 0 0 0 1.5-1 0-2.5 0 0 .5-1.5-1-2.5-.5-2.5-.5-2 .5-3.5 1-2 2.5-2 2.5-4-8.5-1.5-18.5-1.5-27 0z&quot; stroke-linecap=&quot;butt&quot; /&gt;
&lt;path d=&quot;M11.5 30c3.5-1 18.5-1 22 0M12

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.sheridan(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_hzEfj3-jT0Gsb5TKGf1L5A&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1890&quot; height=&quot;220&quot; viewBox=&quot;0 0 1890 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;1600&quot; height=&quot;900&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_0_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect fill=&#x27;#d9ffd9 width=&#x27;1600&#x27; height=&#x27;900&#x27;/&gt;&lt;path  fill=&#x27;#7f7fc9&#x27; d=&#x27;M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z&#x27;/&gt;&lt;path  fill=&#x27;#c97f7f&#x27; d=&#x27;M751.6 450.5c0-49.5 99.8-130.3 99.8-180c0-49.7-99.8-130.5-99.8-180s49.4-90 49.4-90v-0.5h-802v900h793c0 0-40.4-43-40.4-89.5c0-49.5 99.8-130.3 99.8-180s-99.8-130.5-99.8-180z&#x27;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;1600&quot; height=&quot;900&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_1_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect fill=&#x27;#f4edff width=&#x27;1600&#x27; height=&#x27;900&#x27;/&gt;&lt;path  fill=&#x27;#d4beae&#x27; d=&#x27;M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z&#x27;/&gt;&lt;path  fill=&#x27;#aed4be&#x27; d=&#x27;M751.6 450.5c0-49.5 99.8-130.3 99.8-180c0-49.7-99.8-130.5-99.8-180s49.4-90 49.4-90v-0.5h-802v900h793c0 0-40.4-43-40.4-89.5c0-49.5 99.8-130.3 99.8-180s-99.8-130.5-99.8-180z&#x27;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;1600&quot; height=&quot;900&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;CountryStar_2_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect fill=&#x27;#ffe6cf width=&#x27;1600&#x27; height=&#x27;900&#x27;/&gt;&lt;path  fill=&#x27;#86fdc0&#x27; d=&#x27;M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z&#x27;/&gt;&lt;path  fill=&#x27;#c086fd&#x27; d

So we have a bunch of flag patterns (swirl, yin, sheridan), but ultimately we need a dispatch function using the patternIndex for a flag. Can you build? any other interesing kind patterns to use

In [ ]:
#| export
@patch
def chevron(self:CountryFlag, id):
    """Zigzag chevron pattern"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <path fill='{self.comp}' d='M0 0 L100 50 L200 0 L200 50 L100 100 L0 50Z'/>
    <path fill='{self.comp}' d='M0 100 L100 150 L200 100 L200 150 L100 200 L0 150Z'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.25)'
    return pat

@patch
def scales(self:CountryFlag, id):
    """Fish-scale / overlapping semicircle pattern"""
    content = f"""<rect fill='{self.primary}' width='100' height='100'/>
    <circle cx='50' cy='100' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='0' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='100' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>"""
    pat = SVGDef("pattern", id, content, width=100, height=100, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.35)'
    return pat

@patch
def diamonds(self:CountryFlag, id):
    """Argyle diamond grid"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <polygon points='100,0 200,100 100,200 0,100' fill='{self.comp}'/>
    <line x1='0' y1='0' x2='200' y2='200' stroke='{self.tri1}' stroke-width='4'/>
    <line x1='200' y1='0' x2='0' y2='200' stroke='{self.tri1}' stroke-width='4'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.2)'
    return pat


In [ ]:
#| export
@patch
def fanBlade(self:CountryFlag, id):
    blade_id = f"{id}_b"
    content = f"""<rect width='100' height='100' fill='{self.primary}'/>
    <g stroke='none'>
        <path id='{blade_id}' fill='{self.comp}' d='M75 50c-6.9 0-12.5-5.6-12.5-12.5S68.1 25 75 25c0-6.9-5.6-12.5-12.5-12.5S50 18.1 50 25s-5.6 12.5-12.5 12.5S25 31.9 25 25c-6.9 0-12.5 5.6-12.5 12.5S18.1 50 25 50s12.5 5.6 12.5 12.5S31.9 75 25 75c0 6.9 5.6 12.5 12.5 12.5S50 81.9 50 75s5.6-12.5 12.5-12.5S75 68.1 75 75c6.9 0 12.5-5.6 12.5-12.5S81.9 50 75 50z'/>
        <use href='#{blade_id}' x='-50' y='50'/>
        <use href='#{blade_id}' x='-50' y='-50'/>
        <use href='#{blade_id}' x='50' y='-50'/>
        <use href='#{blade_id}' x='50' y='50'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=100, height=100,
                  patternUnits="userSpaceOnUse")
    scale = 0.4
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def sheridan(self:CountryFlag, id):
    W, H = 200, 200
    A = 30  # wave amplitude
    cx1, cx2 = 50, 150  # wave boundary centers

    content = f"""<rect width='{W}' height='{H}' fill='{self.primary}'/>
    <path fill='{self.comp}' d='
        M {cx1},0
        Q {cx1+A},{H//4} {cx1},{H//2}
        Q {cx1-A},{3*H//4} {cx1},{H}
        L {cx2},{H}
        Q {cx2-A},{3*H//4} {cx2},{H//2}
        Q {cx2+A},{H//4} {cx2},0
        Z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=W, height=H,
                  patternUnits="userSpaceOnUse")
    scale = 0.5
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def leafy(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='250' height='200'/>
    <g fill='{self.comp}'>
        <path d='M161 100c0 33.22-36 34.14-36 60c0-25.86-36-26.78-36-60s36-34.14 36-60C125 65.86 161 66.78 161 100z'/>
        <path d='M35 0C35 33.22 0 34.14 0 60c0-25.86-65-90-65-90S35-33.22 35 0z'/>
        <path d='M35 200c0 33.22-100 30-100 30s65-64.14 65-90C0 165.86 35 166.78 35 200z'/>
        <path d='M315-30c0 0-65 64.14-65 90c0-25.86-35-26.78-35-60S315-30 315-30z'/>
        <path d='M315 230c0 0-100 3.22-100-30s35-34.14 35-60C250 165.86 315 230 315 230z'/>
    </g>
    <g fill='none' stroke='{self.tri1}' stroke-width='40'>
        <path d='M39.53-98.77c0 40.43 46.61 56.95 46.61 98.85s-47.19 58.59-47.19 99.3s47.19 57.24 47.19 99.3s-46.61 58.41-46.61 98.85'/>
        <path d='M211.72 297.53c0-40.71-47.19-57.24-47.19-99.3s46.61-58.41 46.61-98.85c0-40.43-46.61-56.95-46.61-98.85s47.19-58.59 47.19-99.3'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=250, height=200,
                  patternUnits="userSpaceOnUse")
    scale = 0.35
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def angled(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='2000' height='1000'/>
<path fill='{self.comp}' d='M1500 100l-100 100v200l200 200V200zm500 100V0l-200 200v200zM2000 1000l-144-144 144-56zm-200-600l-400 400 200 200V800l400-400zM1200 800v200h-200zm100-300l-300 300h200l200-200zm-200-400L1200 0h-200v200l200 200h200z'/>
<path fill='{self.tri1}' d='M1600 0h-200l200 200v400l200-200V200h-200L1800 0zM2000 600V400l-200 200v200zM800 0H600l200 200 400 400 100-100zm400 0l-100 100 300 300V200zm200 800h-200l200 200h200zM1800 800l-200 200h200l200-200zm-800 0L600 400H400l400 400-200 200h200l200-200zM600 800L400 600V400L200 600 0 800v200h400l100-100zm0-800H200l200 200z'/>
<path fill='{self.comp}' d='M800 200V0L400 400h200l100-100 300 300h200zM0 200V0h200zm600 400L400 400v200l200 200h200zM400 200L300 100 0 400l200 200V400zm100 700L200 600 100 700l300 300zm-300 100H0V800z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=2000, height=1000,
                  patternUnits="userSpaceOnUse")
    scale = 0.08
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
def _settlement_pattern(pat_id: str, flag: 'CountryFlag', size: int = 40) -> str:
    """Generate an SVG <pattern> element from a CountryFlag's colors + patternIndex."""
    s = size
    h = s // 2
    p, c, t1, t2 = flag.primary, flag.comp, flag.tri1, flag.tri2
    idx = flag.patternIndex % 6

    if idx == 0:    # horizontal bicolor
        inner = (f'<rect width="{s}" height="{h}" fill="{p}"/>'
                 f'<rect y="{h}" width="{s}" height="{h}" fill="{c}"/>')
    elif idx == 1:  # vertical bicolor
        inner = (f'<rect width="{h}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{s}" fill="{c}"/>')
    elif idx == 2:  # tricolor horizontal
        t = s // 3
        inner = (f'<rect width="{s}" height="{t}" fill="{p}"/>'
                 f'<rect y="{t}" width="{s}" height="{t}" fill="{t1}"/>'
                 f'<rect y="{2*t}" width="{s}" height="{t}" fill="{c}"/>')
    elif idx == 3:  # quarters
        inner = (f'<rect width="{h}" height="{h}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{h}" fill="{c}"/>'
                 f'<rect y="{h}" width="{h}" height="{h}" fill="{t1}"/>'
                 f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{t2}"/>')
    elif idx == 4:  # cross on solid
        w = s // 5
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{c}"/>'
                 f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{c}"/>')
    else:           # diagonal split
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<polygon points="0,0 {s},0 0,{s}" fill="{c}"/>')

    return (f'<pattern id="{pat_id}" width="{s}" height="{s}" '
            f'patternUnits="userSpaceOnUse">{inner}</pattern>')


In [ ]:
#| export
@patch
def bicolorH(self:CountryFlag, id):
    """Horizontal bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{s}" height="{h}" fill="{self.primary}"/><rect y="{h}" width="{s}" height="{h}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def bicolorV(self:CountryFlag, id):
    """Vertical bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{h}" height="{s}" fill="{self.primary}"/><rect x="{h}" width="{h}" height="{s}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def tricolorH(self:CountryFlag, id):
    """Horizontal tricolor flag"""
    s = 42; t = s // 3
    content = (f'<rect width="{s}" height="{t}" fill="{self.primary}"/>'
               f'<rect y="{t}" width="{s}" height="{t}" fill="{self.tri1}"/>'
               f'<rect y="{2*t}" width="{s}" height="{t}" fill="{self.comp}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def quarters(self:CountryFlag, id):
    """Quartered flag"""
    s = 40; h = s // 2
    content = (f'<rect width="{h}" height="{h}" fill="{self.primary}"/>'
               f'<rect x="{h}" width="{h}" height="{h}" fill="{self.comp}"/>'
               f'<rect y="{h}" width="{h}" height="{h}" fill="{self.tri1}"/>'
               f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
#| export

@patch
def crossFlag(self:CountryFlag, id):
    """Cross on solid background with center circle in tri2"""
    s = 40; h = s // 2; w = s // 5; r = w 
    content = (f'<rect width="{s}" height="{s}" fill="{self.primary}"/>'
               f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{self.comp}"/>'
               f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{self.comp}"/>'
               f'<circle cx="{h}" cy="{h}" r="{r}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def flagPattern(self:CountryFlag, id, scale=None):
    """Dispatch to pattern based on patternIndex, with optional scale override"""
    patterns = [
        self.circlePattern,  # 0
        self.triPattern,     # 1
        self.swirl,          # 2
        self.yin,            # 3
        self.weave,          # 4
        self.sheridan,       # 5
        self.chevron,        # 6
        self.scales,         # 7
        self.diamonds,       # 8
        self.fanBlade,       # 9
        self.leafy,          # 10
        self.bicolorH,       # 11
        self.bicolorV,       # 12
        self.tricolorH,      # 13
        self.quarters,       # 14
        self.crossFlag,      # 15
    ]
    idx = self.patternIndex % len(patterns)
    pat = patterns[idx](id)
    
    if scale is not None:
        # Parse existing scale from patternTransform (default 1.0 if not set)
        existing_transform = pat.attributes.get('patternTransform', 'scale(1)')
        m = re.search(r'scale\(([\d.]+)\)', existing_transform)
        base_scale = float(m.group(1)) if m else 1.0
        pat.attributes['patternTransform'] = f'scale({base_scale * scale})'
    
    return pat


Can we update flagPattern to take a scale parameters. sometimes we are going to need to shrink the flag

So the tricky part is that each of the underlying flags have their own scale. it needs to read the existing scale and then modify it

## Packets

Can you make the gallery have 3 rows

## Gallery

In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60  # hex height + label space

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    sampleHex = Hex(radius=radius, center=MapCord(cx, cy), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_guvr6rlHQuCnsRSb5G0U3Q&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1080&quot; height=&quot;680&quot; viewBox=&quot;0 0 1080 680&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;120&quot; height=&quot;90&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_0_pat&quot;&gt;&lt;rect width=120 height= 90 fill=&quot;#27564e&quot;/&gt;&lt;circle cx=&quot;50&quot; cy=&quot;45&quot; r=&quot;30&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_1_pat&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#ffc8d1&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#27564e&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#89f771&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#7189f7&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_2_pat&quot; patternTransform=&quot;scale(0.2)&quot;&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#ffc8d1&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#27564e&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;
&lt;/pattern&gt;
&lt;pattern width=&quot;800&quot; 

Can you update the gallery so it uses king pieces instead of hexes

@patch
def kingPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, piece_id: str = None, fill: str = None):
    """Place a chess king at center, colored with CountryFlag colors.
    The raw piece is drawn in a 45x45 box centered ~(22.5, 22.5).
    fill: override body fill (e.g. 'url(#pattern_id)' for pattern fill)"""
    
    if piece_id is None:
        piece_id = f"king_{self.name}"
    if fill is None:
        fill = self.primary
    
    paths = f'''<path d="M22.5 11.63V6M20 8h5" stroke-linejoin="miter" />
<path d="M22.5 25s4.5-7.5 3-10.5c0 0-1-2.5-3-2.5s-3 2.5-3 2.5c-1.5 3 3 10.5 3 10.5"
      fill="{fill}" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M11.5 37c5.5 3.5 15.5 3.5 21 0v-7s9-4.5 6-10.5c-4-6.5-13.5-3.5-16 4V27v-3.5c-3.5-7.5-13-10.5-16-4-3 6 5 10 5 10V37z"
      fill="{fill}" />
<path d="M11.5 30c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0" />'''

    tx, ty = center.x, center.y
    
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)">
{paths}
</g>'''


In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

# Background rects so the flag pattern shows behind the king
for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName,scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    # Draw a colored circle as backdrop, then the king on top
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stoke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    
    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_pSB2c9Y6Qc_wE9WM4F6C2g&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1080&quot; height=&quot;680&quot; viewBox=&quot;0 0 1080 680&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;120&quot; height=&quot;90&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_0_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect width=120 height= 90 fill=&quot;#27564e&quot;/&gt;&lt;circle cx=&quot;50&quot; cy=&quot;45&quot; r=&quot;30&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_1_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#ffc8d1&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#27564e&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#89f771&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#7189f7&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_2_pat&quot; patternTransform=&quot;scale(0.020000000000000004)&quot;&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#ffc8d1&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#27564e&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#

Can we do the label for the demo using the flag labelStyle?

Can you rewrite it?

In [ ]:
??SVGBuilder.add_font


```python
@patch
def add_font(self: SVGBuilder, font_name: str, url: str = None):
    """Add a web font to the SVG. Defaults to Google Fonts if no URL given.

    Args:
        font_name: Font family name (e.g. 'PT Sans', 'Cinzel')
        url: Full CSS URL. If None, auto-generates a Google Fonts import URL.
    """
    if not hasattr(self, 'fonts'):
        self.fonts = {}

    if url is None:
        encoded = font_name.replace(' ', '+')
        url = f"https://fonts.googleapis.com/css2?family={encoded}&display=swap"

    self.fonts[font_name] = url
    return self
```

**File:** `~/HexMagic/HexMagic/styles.py`

In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding
canvas.add_font('Cinzel')

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName, scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stroke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    # Label using flag's labelStyle
    lbl_style = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl_style)
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="\'Cinzel\', sans-serif" class="{lbl_style.name}">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_ppdzl7bUStSXEJtlOZQrag&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1080&quot; height=&quot;680&quot; viewBox=&quot;0 0 1080 680&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;120&quot; height=&quot;90&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_0_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect width=120 height= 90 fill=&quot;#27564e&quot;/&gt;&lt;circle cx=&quot;50&quot; cy=&quot;45&quot; r=&quot;30&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;96&quot; height=&quot;96&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_1_pat&quot; patternTransform=&quot;scale(0.1)&quot;&gt;&lt;rect width=96 height= 96 fill=&quot;#ffc8d1&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;48&quot; fill=&quot;#27564e&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;32&quot; fill=&quot;#89f771&quot;/&gt;
        &lt;circle cx=&quot;48&quot; cy=&quot;48&quot; r=&quot;16&quot; fill=&quot;#7189f7&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;400&quot; height=&quot;400&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;gallery_2_pat&quot; patternTransform=&quot;scale(0.020000000000000004)&quot;&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;rect width=400 height=400 fill=&#x27;#ffc8d1&#x27; /&gt;&lt;/g&gt;
&lt;g  fill=&#x27;#27564e&#x27; fill-opacity=&#x27;1&#x27;&gt;&lt;path d=&#x27;M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;1&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;200&#x27; cy=&#x27;400&#x27; r=&#x27;60&#x27;/&gt;&lt;/g&gt;&lt;g  fill=&#x27;#ffc8d1&#x27;&gt;&lt;circle  cx=&#x27;0&#x27; cy=&#x27;200&#x27; r=&#x27;60&#x27;/&gt;&lt;circle  cx=&#x27;400&#x27; cy=&#

So the following is some downstream code which I think should be improved since I think we could have a factory function from the flag that is put onto a piece. I do think we are going to have to come up with what works at various sizes. I love the patterns so far for the larger pieces, but we are going to need another for items in a list and another for a piece on a board (say 20x20 or smaller).

```
_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_PIECE_ICONS = {
    PieceType.PAWN:   "♟", PieceType.KNIGHT: "♞", PieceType.ROOK:   "♜",
    PieceType.BISHOP: "♝", PieceType.QUEEN:  "♛", PieceType.KING:   "♚",
}


@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    """Register flag pattern on grid builder, return CSS fill reference.
    Falls back to flag.primary solid color if no flag."""
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    pat = self.flag.flagPattern(pat_name, scale=pat_scale)
    grid.builder.add_definition(pat)
    return f"url(#{pat_name})"


@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             pat_fill: str = None, suffix: str = "") -> str:
    """Render this piece's chess glyph as SVG at a hex position.

    Args:
        grid:     HexGrid to draw on (needed for hex centres + builder)
        hex_idx:  fine grid index to draw at (defaults to self.location)
        scale:    piece scale (defaults to ~75% of hex diameter)
        opacity:  1.0 for solid, 0.4 for ghost, etc.
        pat_fill: override the fill (otherwise registers flag pattern)
        suffix:   appended to piece_id for uniqueness (e.g. '_ghost')
    """
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes): return ""

    center = grid.hexes[idx].center
    renderer = _PIECE_RENDERERS.get(self.piece_type, 'pawnPiece')
    if not self.flag: return ""

    if scale is None:
        scale = (grid.radius * 0.75) / 22.5  # chess SVGs ~22.5px baseline

    fill = pat_fill or self._ensure_pattern(grid)
    pid  = f"piece_{self.id[:8]}{suffix}"

    svg = getattr(self.flag, renderer)(
        MapCord(center.x, center.y),
        scale=scale,
        piece_id=pid,
        fill=fill,
    )

    # Wrap in a group with opacity if not fully opaque
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>\n'

    return svg


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    """Draw a faded copy at a projected future position."""
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")
```

### Piece Plan

In [ ]:
#| export
@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board',
               scale: float = 1.0) -> tuple:
    if size == 'board':
        return self.primary, None

    if size == 'list':
        _SIMPLE = [
            self.bicolorH, self.bicolorV, self.tricolorH,
            self.quarters, self.crossFlag, self.chevron,
        ]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        pat = fn(pat_id)
        # Scale tiles to match rendered piece size (~45 unit piece × scale = pixels)
        # This gives ~1 clean tile repetition across the piece
        pat.attributes['patternTransform'] = f'scale({scale:.2f})'
        return f"url(#{pat_id})", pat

    # 'large'
    pat = self.flagPattern(pat_id, scale=0.1)
    return f"url(#{pat_id})", pat


In [ ]:
#| export
@patch
def _piece_renderer(self: CountryFlag, piece_type: PieceType):
    """Return the bound method for drawing a given piece type."""
    return {
        PieceType.KING:   self.kingPiece,
        PieceType.QUEEN:  self.queenPiece,
        PieceType.BISHOP: self.bishopPiece,
        PieceType.KNIGHT: self.knightPiece,
        PieceType.ROOK:   self.rookPiece,
        PieceType.PAWN:   self.pawnPiece,
    }[piece_type]


@patch
def piece_svg(self: CountryFlag, piece_type: PieceType, center: MapCord,
              scale: float = 1.0, size: str = 'board',
              piece_id: str = None, attrs: dict = None) -> tuple:
    """Render a chess piece SVG with appropriate fill for the display tier.

    Returns (svg_string, pattern_def | None).
    Caller registers the def on their SVGBuilder / grid.builder if not None.

    Args:
        piece_type: PieceType enum value
        center:     MapCord placement position
        scale:      SVG scale factor (auto-sized from hex radius when None)
        size:       'board' | 'list' | 'large'
        piece_id:   unique SVG element id  (auto-generated if None)
        attrs:      dict of HTMX / HTML attributes for interactivity
    """
    if piece_id is None:
        piece_id = f"{piece_type.value}_{self.name}"

    pat_id = f"{piece_id}_pat"
    #fill, pat_def = self.piece_fill(pat_id, size=size)
    fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)
    svg = self._piece_renderer(piece_type)(
        center, scale=scale, piece_id=piece_id, fill=fill, attrs=attrs
    )
    return svg, pat_def


@patch
def draw_piece(self: CountryFlag, piece_type: PieceType, center: MapCord,
               builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
               piece_id: str = None, layer: str = None,
               opacity: float = 1.0, attrs: dict = None) -> str:
    """Convenience: render piece, register pattern on builder, return SVG.

    If layer is given, also calls builder.adjust(layer, svg).
    """
    svg, pat_def = self.piece_svg(
        piece_type, center, scale=scale, size=size,
        piece_id=piece_id, attrs=attrs,
    )
    if pat_def is not None:
        builder.add_definition(pat_def)
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


can you write it out. I think these functions should take a PieceType so we don't have _PIRECE_RENDERERS

Can you build a gallery of the different sizes for the queen and a random country?
    

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.QUEEN, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_KjoJSBW3TVGzlN9QZRsOlg&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;580&quot; height=&quot;220&quot; viewBox=&quot;0 0 580 220&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;40&quot; height=&quot;40&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;queen_list_1_pat&quot; patternTransform=&quot;scale(1.50)&quot;&gt;&lt;rect width=&quot;20&quot; height=&quot;40&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;rect x=&quot;20&quot; width=&quot;20&quot; height=&quot;40&quot; fill=&quot;#27564e&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;100&quot; height=&quot;100&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;queen_large_2_pat&quot; patternTransform=&quot;scale(0.034999999999999996)&quot;&gt;&lt;rect fill=&#x27;#ffc8d1&#x27; width=&#x27;100&#x27; height=&#x27;100&#x27;/&gt;
    &lt;circle cx=&#x27;50&#x27; cy=&#x27;100&#x27; r=&#x27;50&#x27; fill=&#x27;none&#x27; stroke=&#x27;#27564e&#x27; stroke-width=&#x27;12&#x27;/&gt;
    &lt;circle cx=&#x27;0&#x27; cy=&#x27;50&#x27; r=&#x27;50&#x27; fill=&#x27;none&#x27; stroke=&#x27;#27564e&#x27; stroke-width=&#x27;12&#x27;/&gt;
    &lt;circle cx=&#x27;100&#x27; cy=&#x27;50&#x27; r=&#x27;50&#x27; fill=&#x27;none&#x27; stroke=&#x27;#27564e&#x27; stroke-width=&#x27;12&#x27;/&gt;&lt;/pattern&gt;
  &lt;/defs&gt;
  &lt;style&gt;
@import url(&#x27;https://fonts.googleapis.com/css2?family=Cinzel&amp;display=swap&#x27;);
.contrast_lbl_2 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_1 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_0 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;bg_0&quot;&gt;
&lt;circle cx=&quot;80&quot; cy=&quot;90&quot; r=&quot;20.5&quot; fill=&quot;white&quot; stroke=&quot;#27564e&quot; stroke-width=&quot;1.5&quot;/&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;queen_0&quot;&gt;
&lt;g id=&quot;queen_board_0&quot; fill=&quot;#ffc8d1&quot; fill-rule=&quot;evenodd&quot;
   stroke=&quot;#27564e&quot; stroke-width=&quot;1.5&quot;
   stroke-linecap=&quot;round&quot; stroke-linejo

There seems to be some wrong colors for the middle one of list at 1.5 is it a scale issue?

In [ ]:
#!cat ../../docs/*.md

from dataclasses import dataclass, field
from typing import Optional, List, Set
from enum import Enum
import uuid

class PieceGoal(Enum):
    """Available goals for pieces"""
    EXPLORE = "explore"
    HARVEST = "harvest"
    ATTACK = "attack"
    MOVE = "move"
    SETTLE = "settle"
    DEFEND = "defend"
    PATROL = "patrol"

class PersonalityTrait(Enum):
    """Personality tendencies when acting autonomously"""
    AGGRESSIVE = "aggressive"      # Prefers attack/explore
    DEFENSIVE = "defensive"        # Prefers defend/settle
    ECONOMIC = "economic"          # Prefers harvest/settle
    EXPLORER = "explorer"          # Prefers explore/move
    BALANCED = "balanced"          # No strong preference




@dataclass
class Piece:
    """A game piece representing a group of units."""
    
    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0  # Kingdom/country ID
    parent_id: Optional[str] = None  # ID of piece that spawned this
    
    # Core attributes
    size: int = 100  # Number of units in this piece
    health: int = 100  # Hit points (0-100 scale)
    max_health: int = 100
    
    # Vision and intelligence
    sight: int = 3  # How many hex rings they can see
    memory: float = 0.8  # Retention rate (0-1, higher = better memory)
    
    # Movement
    movement_range: int = 4  # Max weighted hexes per turn
    current_position: int = -1  # Current hex index
    
    # Goals and targeting
    goal: PieceGoal = PieceGoal.EXPLORE
    target_position: Optional[int] = None  # Target hex index
    target_piece: Optional[str] = None  # Target piece UUID
    
    # Behavior
    personality: PersonalityTrait = PersonalityTrait.BALANCED
    
    # Relationships
    spawned_pieces: List[str] = field(default_factory=list)  # UUIDs of children
    
    # Knowledge (what hexes this piece knows about)
    known_hexes: Set[int] = field(default_factory=set)
    knowledge_freshness: dict = field(default_factory=dict)  # hex_idx -> turn_last_seen
    
    # Settlement state
    is_settled: bool = False
    settle_progress: int = 0  # Turns spent settling (settlement complete at threshold)
    settle_threshold: int = 3


In [ ]:
import os
import glob

def concatenate_text_files(directory_path):
    # Change the current working directory to the specified path
    os.chdir(directory_path)
    
    # Use glob to find all files matching a pattern (e.g., all .txt files)
    # Adjust '*.txt' to match your specific file extension if needed
    file_list = glob.glob('*.py') 
    
    # Open the output file in write mode ('w')
    ret = ""
    for fname in file_list:
        print(fname)
        with open(fname, 'r') as infile:
            # Read the content and write it to the output file
            ret += f"{fname}\n\n " + infile.read() 
            # Optional: Add a newline character as a separator between files
            #outfile.write('\n')
    return ret

primitives = concatenate_text_files("../../HexMagic/plot")

does $`primitives` have information about HexRegions

rootIdeas  = concatenate_text_files("../../HexMagic")

in $`rootIdeas` do you see how terrain has encode and decode. can we build something similar for piece

In [ ]:
from HexMagic.core import Terrain

In [ ]:
??Terrain.encode


```python
def encode(self):
    grid = self.hexGrid 
    nRows = grid.nRows
    nCols = grid.nCols
    ret = f"radius:{self.hexGrid.radius}\n"
    ret += f"size:{self.hexGrid.nRows}^{self.hexGrid.nCols}\n"
    ret += f"path:{self.path}\n"
    field_names = '\t'.join(self.fields.keys())
    ret += f"fields:{field_names}\n"


    # Add geobounds if present
    if self.geo is not None:
        ret += f"geo:{self.geo.lat_min},{self.geo.lat_max},{self.geo.lon_min},{self.geo.lon_max}\n"

    # Add climate if present
    if self.climate is not None:
        ret += f"climate:{self.climate.encode()}\n"

    ret += f"+data:\n"
    i = 0
    for row in range(nRows):
        line = []
        for col in range(nCols):
            csv_parts = [str(self.elevations[i])]
            for fieldName in self.fields.keys():
                csv_parts.append(str(self.fields[fieldName][i]))
            line.append(','.join(csv_parts))
            i += 1
        ret += "\t".join(line) + "\n"
    ret += f"-data:\n"

    return ret
```

**File:** `~/HexMagic/HexMagic/terrain.py`

Does the terrain style encoding make sense to work for piece

So we have a bunch of patterns for the flag ( sheridan, weave, yin, swirl, triPattern ) but ultimately the country should just have one pattern. can we build a function that will assign one pattern. This will need to be consistent (ie encode and decode) so maybe it is an int index into a function 

Can you write it?

In [ ]:
#| export
@patch
def flag_banner(self: CountryFlag, width=150, height=100,
                pattern_fn=None, wavy=False, pole=True, attrs=None):
    """Traditional rectangular flag with optional pole and wavy edge.
    
    Returns SVGBuilder. Use .show() for Jupyter or wrap for FastHTML:
        Div(NotStr(flag.flag_banner().xml()), **htmx_attrs)
    
    Args:
        width/height: flag dimensions
        pattern_fn: 'swirl','weave','triPattern','circlePattern','yin','sheridan'
                    (defaults via patternIndex)
        wavy: wavy right edge via MapPath.make_windy
        pole: draw flagpole on left
        attrs: extra SVG attributes dict for the flag polygon
    """
    PATTERNS = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']
    if pattern_fn is None:
        pattern_fn = PATTERNS[self.patternIndex % len(PATTERNS)]
    
    pad, pole_w, finial_r, pole_extra = 6, 5, 6, 30
    
    # Layout offsets
    fx = pad + (pole_w if pole else 0)
    fy = pad + (finial_r * 2 if pole else 0)
    
    builder = SVGBuilder()
    builder.width = fx + width + pad
    builder.height = fy + height + (pole_extra if pole else 0) + pad
    
    # --- Pattern fill ---
    pat_id = f"fp_{self.name}"
    builder.add_definition(getattr(self, pattern_fn)(pat_id))
    flag_style = StyleCSS(f"flag_{self.name}",
                          fill=f"url(#{pat_id})", stroke=self.comp, stroke_width=2.5)
    builder.add_style(flag_style)
    
    # --- Flag shape via MapPath ---
    tl, tr = MapCord(fx, fy), MapCord(fx + width, fy)
    br, bl = MapCord(fx + width, fy + height), MapCord(fx, fy + height)
    
    if wavy:
        right_edge = MapPath([tr, br]).make_windy(iterations=3, offset_factor=0.12)
        points = [tl, tr] + right_edge.points[1:] + [bl]
    else:
        points = [tl, tr, br, bl]
    
    flag_path = MapPath(points, flag_style)
    builder.adjust("flag", flag_path.drawClosed(CountryFlag._render_attrs(attrs)))
    
    # --- Pole + finial ---
    if pole:
        pole_style = StyleCSS(f"pole_{self.name}",
                              stroke=self.darkPrimary, stroke_width=pole_w,
                              stroke_linecap="round", fill=self.darkPrimary)
        builder.add_style(pole_style)
        px = pad + pole_w / 2
        builder.adjust("pole",
            f'<line x1="{px}" y1="{pad + finial_r}" '
            f'x2="{px}" y2="{fy + height + pole_extra}" '
            f'class="{pole_style.name}"/>'
            f'<circle cx="{px}" cy="{pad + finial_r}" '
            f'r="{finial_r}" class="{pole_style.name}"/>')
    
    return builder


I need a flag_svg much like there is a piece_svg. can you do that and build a gallery?

In [ ]:
#| export
@patch
def flag_svg(self: CountryFlag, center: MapCord, scale: float = 1.0,
             size: str = 'board', flag_id: str = None,
             wavy: bool = False, pole: bool = True,
             attrs: dict = None) -> tuple:
    """Render a flag SVG fragment with appropriate fill for the display tier.

    Returns (svg_string, pattern_def | None).
    Caller registers the def on their SVGBuilder if not None.

    Base flag is 30×20 units, centered at `center`, scaled by `scale`.

    Args:
        center:  MapCord placement position
        scale:   SVG scale factor
        size:    'board' | 'list' | 'large'
        flag_id: unique SVG element id
        wavy:    wavy trailing edge
        pole:    include flagpole + finial
        attrs:   dict of HTMX / HTML attributes
    """
    if flag_id is None:
        flag_id = f"flag_{self.name}"

    pat_id = f"{flag_id}_pat"
    pat_def = None

    # --- Fill tier ---
    if size == 'board':
        fill = self.primary
    elif size == 'list':
        _SIMPLE = [self.bicolorH, self.bicolorV, self.tricolorH,
                   self.quarters, self.crossFlag, self.chevron]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        pat_def = fn(pat_id)
        pat_def.attributes['patternTransform'] = f'scale({scale:.3f})'
        fill = f"url(#{pat_id})"
    else:  # 'large'
        pat_def = self.flagPattern(pat_id, scale=0.1 * scale)
        fill = f"url(#{pat_id})"

    # --- Base dimensions (local coords) ---
    fw, fh = 30, 20
    pole_w, finial_r = 1.2, 1.5
    pole_pad = (pole_w + 0.5) if pole else 0

    parts = []
    extra = CountryFlag._render_attrs(attrs) if attrs else ""

    # --- Flag shape ---
    x0, y0 = pole_pad, 0
    if wavy:
        tr, br = MapCord(x0 + fw, y0), MapCord(x0 + fw, fh)
        right = MapPath([tr, br]).make_windy(iterations=3, offset_factor=0.12)
        d = f"M {x0},{y0} L {x0 + fw},{y0} "
        for p in right.points[1:]:
            d += f"L {p.x:.1f},{p.y:.1f} "
        d += f"L {x0},{fh} Z"
        parts.append(f'<path d="{d}" fill="{fill}" stroke="{self.comp}" '
                     f'stroke-width="0.7"{extra}/>')
    else:
        parts.append(f'<rect x="{x0}" y="{y0}" width="{fw}" height="{fh}" '
                     f'fill="{fill}" stroke="{self.comp}" stroke-width="0.7" '
                     f'rx="0.5"{extra}/>')

    # Board-tier accent stripe so tiny flags still read as two-tone
    if size == 'board':
        sw = fw * 0.3
        parts.append(f'<rect x="{x0}" y="{y0}" width="{sw:.1f}" height="{fh}" '
                     f'fill="{self.comp}" opacity="0.85"/>')

    # --- Pole + finial ---
    if pole:
        px = pole_w / 2
        parts.append(
            f'<line x1="{px}" y1="-{finial_r}" x2="{px}" y2="{fh + 2}" '
            f'stroke="{self.darkPrimary}" stroke-width="{pole_w}" '
            f'stroke-linecap="round"/>')
        parts.append(
            f'<circle cx="{px}" cy="-{finial_r}" r="{finial_r}" '
            f'fill="{self.darkPrimary}"/>')

    inner = '\n'.join(parts)

    # Center the whole thing at origin, then place at center
    total_w = pole_pad + fw
    cx_off = total_w / 2
    cy_off = fh / 2
    tx, ty = center.x, center.y

    svg = (f'<g id="{flag_id}" '
           f'transform="translate({tx},{ty}) scale({scale}) '
           f'translate(-{cx_off:.1f},-{cy_off:.1f})" '
           f'style="cursor:pointer">\n{inner}\n</g>')

    return svg, pat_def


@patch
def draw_flag(self: CountryFlag, center: MapCord, builder: SVGBuilder,
              scale: float = 1.0, size: str = 'board',
              flag_id: str = None, layer: str = None,
              wavy: bool = False, pole: bool = True,
              opacity: float = 1.0, attrs: dict = None) -> str:
    """Convenience: render flag, register pattern on builder, return SVG.
    If layer is given, also calls builder.adjust(layer, svg).
    """
    svg, pat_def = self.flag_svg(
        center, scale=scale, size=size, flag_id=flag_id,
        wavy=wavy, pole=pole, attrs=attrs)
    if pat_def is not None:
        builder.add_definition(pat_def)
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


In [ ]:
import random

# A few countries with different pattern indices
flags = CountryFlag.seaborn("husl", 4)
for i, f in enumerate(flags):
    f.patternIndex = i * 3  # spread across pattern types

tiers = [
    ('board', 0.8,  False, 'solid + stripe'),
    ('board', 1.5,  True,  'board wavy'),
    ('list',  1.5,  False, 'simple pattern'),
    ('list',  2.5,  True,  'list wavy'),
    ('large', 3.0,  False, 'full pattern'),
    ('large', 4.0,  True,  'large wavy'),
]

cols = len(tiers)
rows = len(flags)
col_w = 160
row_h = 130

canvas = SVGBuilder()
canvas.add_font('Cinzel')
canvas.width = cols * col_w + 40
canvas.height = rows * row_h + 60

# Column headers
for j, (sz, sc, wavy, desc) in enumerate(tiers):
    cx = 40 + j * col_w + col_w // 2
    canvas.adjust(f"hdr_{j}",
        f'<text x="{cx}" y="20" text-anchor="middle" font-size="11" '
        f'font-family="\'Cinzel\', sans-serif" fill="#333">{desc}</text>'
        f'<text x="{cx}" y="34" text-anchor="middle" font-size="10" '
        f'font-family="sans-serif" fill="#888">{sz} ×{sc}</text>')

for i, flag in enumerate(flags):
    # Row label
    ry = 55 + i * row_h + row_h // 2
    lbl = flag.labelStyle(f"rlbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"rlbl_{i}",
        f'<text x="5" y="{ry}" font-size="11" font-family="\'Cinzel\', sans-serif" '
        f'class="{lbl.name}">{flag.name}</text>')

    for j, (sz, sc, wavy, desc) in enumerate(tiers):
        cx = 40 + j * col_w + col_w // 2
        cy = 55 + i * row_h + row_h // 2

        fid = f"f_{i}_{j}"

        # Backdrop circle
        r = sc * 16
        canvas.adjust(f"bg_{i}_{j}",
            f'<circle cx="{cx}" cy="{cy}" r="{r + 6}" fill="white" '
            f'stroke="{flag.comp}" stroke-width="0.8" opacity="0.4"/>')

        flag.draw_flag(
            MapCord(cx, cy), canvas,
            scale=sc, size=sz, flag_id=fid, layer=f"flag_{i}_{j}",
            wavy=wavy, pole=(sc >= 1.5),
        )

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_rtUcwY1tRlKiANZj4hp1Sw&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;1000&quot; height=&quot;580&quot; viewBox=&quot;0 0 1000 580&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;defs&gt;
    &lt;pattern width=&quot;40&quot; height=&quot;40&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;f_0_2_pat&quot; patternTransform=&quot;scale(1.500)&quot;&gt;&lt;rect width=&quot;40&quot; height=&quot;20&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;rect y=&quot;20&quot; width=&quot;40&quot; height=&quot;20&quot; fill=&quot;#27564e&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;40&quot; height=&quot;40&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;f_0_3_pat&quot; patternTransform=&quot;scale(2.500)&quot;&gt;&lt;rect width=&quot;40&quot; height=&quot;20&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;rect y=&quot;20&quot; width=&quot;40&quot; height=&quot;20&quot; fill=&quot;#27564e&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;120&quot; height=&quot;90&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;f_0_4_pat&quot; patternTransform=&quot;scale(0.30000000000000004)&quot;&gt;&lt;rect width=120 height= 90 fill=&quot;#27564e&quot;/&gt;&lt;circle cx=&quot;50&quot; cy=&quot;45&quot; r=&quot;30&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;120&quot; height=&quot;90&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;f_0_5_pat&quot; patternTransform=&quot;scale(0.4)&quot;&gt;&lt;rect width=120 height= 90 fill=&quot;#27564e&quot;/&gt;&lt;circle cx=&quot;50&quot; cy=&quot;45&quot; r=&quot;30&quot; fill=&quot;#ffc8d1&quot;/&gt;&lt;/pattern&gt;
&lt;pattern width=&quot;40&quot; height=&quot;40&quot; patternUnits=&quot;userSpaceOnUse&quot; id=&quot;f_1_2_pat&quot; patternTransform=&quot;scale(1.500)&quot;&gt;&lt;rect width=&quot;20&quot; height=&quot;20&quot; fill=&quot;#dee5a5&quot;/&gt;&lt;rect x=&quot;20&quot; width=&quot;20&quot; height=&quot;20&quot; fill=&quot;#161139&quot;/&gt;&lt;rect y=&quot;20&quot; width=&quot;20&quot; height=&quot;20&quot; fill=&quot;#3197a4&quot;/&gt;&lt;rect x=&quot;20&quot; y=&quot;20&quot; width=&quot;20&quot; height=&quot;20&quot; fill=&quot

### Animals

In [ ]:
import inspect
import copy
import colorsys
from importlib import resources


with resources.files('HexMagic').joinpath('data/patterns/animals/Bear.svg').open() as f:
    fishPattern = f.read()
fishPattern

'<svg id="Outline" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512"><path d="M400,88a32.054,32.054,0,0,0-30.987,24H329.889L295.434,94.772A39.744,39.744,0,0,0,264.9,92.6L230.7,104H128c-30.269,0-56.092,16.128-76.751,47.938-16.264,25.041-25.77,54.52-30.88,74.839a71.741,71.741,0,0,0,1.541,40.251L68.764,407.59A23.97,23.97,0,0,0,91.532,424H152a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L136,370.234V354.667l19.457-25.942c44.319,7.422,80.206,4.79,102.807,1.174a226.88,226.88,0,0,0,31.3-7.159l12.447,80.909A23.869,23.869,0,0,0,325.727,424H384a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L368,370.234V312c0-20.1,5.93-34.686,17.624-43.34,26.708-19.764,75.539-5.161,76.023-5.014a8,8,0,0,0,8.01-1.989l24-24a8,8,0,0,0,1.771-8.628L480,190.459V177.126a39.918,39.918,0,0,0-17.813-33.282L432,123.719V120A32.036,32.036,0,0,0,400,88Zm53.312,69.157A23.949,23.949,0,0,1,464,177.126V192a8,8,0,0,0,.572,2.971l14.041,35.1-16.989,16.989c-14.328-3.586-57.5-11.958-85.476,8.706C360.125,267.6,352,2

I have a bunch of files like this that I would want to use for additional icons using the coutry flag. we might try the three sizes as before. do you think you could take in this string and create an icon?

@patch
def iconPiece(self: CountryFlag, svg_source: str, center: MapCord,
              scale: float = 1.0, piece_id: str = None,
              fill: str = None, stroke: str = None,
              stroke_width: float = 0, attrs: dict = None):
    """Render an arbitrary SVG icon (animal, emblem, etc.) at center.
    
    svg_source: raw SVG string with <svg> tag — viewBox used for sizing.
    Normalized to the same ~45px baseline as chess pieces, so scale
    factors are interchangeable with kingPiece, queenPiece, etc.
    
    stroke_width is in the normalized 45px space (0 = no outline).
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}"
    if fill is None:
        fill = self.primary
    if stroke is None:
        stroke = self.comp if stroke_width > 0 else "none"

    # Parse viewBox for native dimensions
    vb_match = re.search(r'viewBox=["\']([^"\']+)["\']', svg_source)
    if vb_match:
        parts = vb_match.group(1).split()
        vb_x, vb_y = float(parts[0]), float(parts[1])
        vb_w, vb_h = float(parts[2]), float(parts[3])
    else:
        vb_x, vb_y, vb_w, vb_h = 0, 0, 512, 512

    # Extract inner content (strip <svg> wrapper)
    inner = re.sub(r'<svg[^>]*>', '', svg_source)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    # Strip any existing fill/stroke attrs from paths so ours take effect
    inner = re.sub(r'\s*fill="[^"]*"', '', inner)
    inner = re.sub(r'\s*stroke="[^"]*"', '', inner)

    # Normalize to 45px baseline (chess piece compatible)
    norm = 45.0 / max(vb_w, vb_h)
    # Stroke width in native SVG coords
    native_sw = stroke_width / (norm * scale) if stroke_width > 0 else 0

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)

    return f'''<g id="{piece_id}"
   fill="{fill}" stroke="{stroke}" stroke-width="{native_sw:.2f}"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale * norm:.6f}) translate({-(vb_w/2 + vb_x):.1f},{-(vb_h/2 + vb_y):.1f})"
   style="cursor:pointer"{extra}>
{inner}
</g>'''


@patch
def draw_icon(self: CountryFlag, svg_source: str, center: MapCord,
              builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
              piece_id: str = None, layer: str = None,
              opacity: float = 1.0, stroke_width: float = 0,
              attrs: dict = None) -> str:
    """Convenience: render icon with flag-colored fill at the right tier,
    register any pattern on builder, optionally add to a layer.
    
    size: 'board' (solid), 'list' (simple pattern), 'large' (full pattern)
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}_{id(svg_source) % 9999}"

    pat_id = f"{piece_id}_pat"
    fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)

    if pat_def is not None:
        builder.add_definition(pat_def)

    svg = self.iconPiece(
        svg_source, center, scale=scale, piece_id=piece_id,
        fill=fill, stroke_width=stroke_width, attrs=attrs,
    )
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

tiers = [
    ('board', 0.5,  'solid fill'),
    ('list',  1.5,  'simple pattern'),
    ('large', 3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx, cy = 80 + i * spacing, 90
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    flag.draw_icon(fishPattern, MapCord(cx, cy), canvas,
                   scale=scale, size=size,
                   piece_id=f"bear_{size}", layer=f"bear_{i}",
                   stroke_width=0.5)

    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" class="{lbl.name}">{size} (×{scale})</text>')

canvas.show()


Can we do the fill instead of the stroke? I was thinking we could use one of the paths as a clip path. this is how you could have a purple bear.

```svg
<svg id="Outline" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512">
  <defs>
    <clipPath id="bearClip">
      <path d="M400,88a32.054,32.054,0,0,0-30.987,24H329.889L295.434,94.772A39.744,39.744,0,0,0,264.9,92.6L230.7,104H128c-30.269,0-56.092,16.128-76.751,47.938-16.264,25.041-25.77,54.52-30.88,74.839a71.741,71.741,0,0,0,1.541,40.251L68.764,407.59A23.97,23.97,0,0,0,91.532,424H152a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L136,370.234V354.667l19.457-25.942c44.319,7.422,80.206,4.79,102.807,1.174a226.88,226.88,0,0,0,31.3-7.159l12.447,80.909A23.869,23.869,0,0,0,325.727,424H384a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L368,370.234V312c0-20.1,5.93-34.686,17.624-43.34,26.708-19.764,75.539-5.161,76.023-5.014a8,8,0,0,0,8.01-1.989l24-24a8,8,0,0,0,1.771-8.628L480,190.459V177.126a39.918,39.918,0,0,0-17.813-33.282L432,123.719V120A32.036,32.036,0,0,0,400,88Z"/>
    </clipPath>
  </defs>
  <rect width="512" height="512" fill="purple" clip-path="url(#bearClip)"/>
  <path fill="none" stroke="black" stroke-width="2" d="M400,88a32.054,32.054,0,0,0-30.987,24H329.889L295.434,94.772A39.744,39.744,0,0,0,264.9,92.6L230.7,104H128c-30.269,0-56.092,16.128-76.751,47.938-16.264,25.041-25.77,54.52-30.88,74.839a71.741,71.741,0,0,0,1.541,40.251L68.764,407.59A23.97,23.97,0,0,0,91.532,424H152a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L136,370.234V354.667l19.457-25.942c44.319,7.422,80.206,4.79,102.807,1.174a226.88,226.88,0,0,0,31.3-7.159l12.447,80.909A23.869,23.869,0,0,0,325.727,424H384a8,8,0,0,0,8-8V395.532a23.97,23.97,0,0,0-16.411-22.768L368,370.234V312c0-20.1,5.93-34.686,17.624-43.34,26.708-19.764,75.539-5.161,76.023-5.014a8,8,0,0,0,8.01-1.989l24-24a8,8,0,0,0,1.771-8.628L480,190.459V177.126a39.918,39.918,0,0,0-17.813-33.282L432,123.719V120A32.036,32.036,0,0,0,400,88Zm53.312,69.157A23.949,23.949,0,0,1,464,177.126V192a8,8,0,0,0,.572,2.971l14.041,35.1-16.989,16.989c-14.328-3.586-57.5-11.958-85.476,8.706C360.125,267.6,352,286.522,352,312v64a8,8,0,0,0,5.471,7.589l13.058,4.353a7.99,7.99,0,0,1,5.471,7.59V408H325.727a7.957,7.957,0,0,1-7.907-6.783l-13.913-90.433a8.01,8.01,0,0,0-11.045-6.143c-.543.23-55.156,22.812-139.431,7.488A7.994,7.994,0,0,0,145.6,315.2l-24,32A8,8,0,0,0,120,352v24a8,8,0,0,0,5.47,7.589l13.059,4.353a7.99,7.99,0,0,1,5.471,7.59V408H91.532a7.99,7.99,0,0,1-7.59-5.47L37.089,261.969a55.762,55.762,0,0,1-1.2-31.289C44.255,197.4,69.984,120,128,120H232a8,8,0,0,0,2.53-.411l35.425-11.808a23.841,23.841,0,0,1,18.322,1.3l36.145,18.072A7.994,7.994,0,0,0,328,128h48a8,8,0,0,0,8-8,16,16,0,0,1,32,0v8a8,8,0,0,0,3.562,6.656ZM424,192a8,8,0,1,1,8,8A8,8,0,0,1,424,192Z"/>
</svg>
```

@patch
def iconPiece(self: CountryFlag, svg_source: str, center: MapCord,
              scale: float = 1.0, piece_id: str = None,
              fill: str = None, outline: float = 0,
              outline_color: str = None, attrs: dict = None):
    """Render an arbitrary SVG icon (animal, emblem, etc.) at center.
    
    svg_source: raw SVG string with <svg> tag — viewBox used for sizing.
    Normalized to the same ~45px baseline as chess pieces, so scale
    factors are interchangeable with kingPiece, queenPiece, etc.
    
    outline: border thickness as fraction of icon size (e.g. 0.06 = 6%).
             Rendered as a filled copy behind, not a stroke.
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}"
    if fill is None:
        fill = self.primary
    if outline_color is None:
        outline_color = self.comp

    # Parse viewBox for native dimensions
    vb_match = re.search(r'viewBox=["\']([^"\']+)["\']', svg_source)
    if vb_match:
        parts = vb_match.group(1).split()
        vb_x, vb_y = float(parts[0]), float(parts[1])
        vb_w, vb_h = float(parts[2]), float(parts[3])
    else:
        vb_x, vb_y, vb_w, vb_h = 0, 0, 512, 512

    # Extract inner content (strip <svg> wrapper)
    inner = re.sub(r'<svg[^>]*>', '', svg_source)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    inner = re.sub(r'\s*fill="[^"]*"', '', inner)
    inner = re.sub(r'\s*stroke="[^"]*"', '', inner)

    # Normalize to 45px baseline (chess piece compatible)
    norm = 45.0 / max(vb_w, vb_h)
    cx_vb = vb_w / 2 + vb_x
    cy_vb = vb_h / 2 + vb_y
    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)

    base_transform = f"translate({tx},{ty}) scale({scale * norm:.6f}) translate({-cx_vb:.1f},{-cy_vb:.1f})"

    parts = []

    # Outline layer: slightly scaled-up solid fill behind
    if outline > 0:
        border_s = 1.0 + outline
        border_transform = (
            f"translate({tx},{ty}) scale({scale * norm * border_s:.6f}) "
            f"translate({-cx_vb:.1f},{-cy_vb:.1f})"
        )
        parts.append(
            f'<g fill="{outline_color}" stroke="none" '
            f'transform="{border_transform}">\n{inner}\n</g>'
        )

    # Main fill layer
    parts.append(
        f'<g fill="{fill}" stroke="none" '
        f'transform="{base_transform}">\n{inner}\n</g>'
    )

    return f'''<g id="{piece_id}" style="cursor:pointer"{extra}>
{''.join(parts)}
</g>'''


@patch
def draw_icon(self: CountryFlag, svg_source: str, center: MapCord,
              builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
              piece_id: str = None, layer: str = None,
              opacity: float = 1.0, outline: float = 0,
              outline_color: str = None, attrs: dict = None) -> str:
    """Convenience: render icon with flag-colored fill at the right tier,
    register any pattern on builder, optionally add to a layer.
    
    size: 'board' (solid), 'list' (simple pattern), 'large' (full pattern)
    outline: border as fraction of size (0.06 = 6% border), fill-based not stroke.
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}_{id(svg_source) % 9999}"

    pat_id = f"{piece_id}_pat"
    fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)

    if pat_def is not None:
        builder.add_definition(pat_def)

    svg = self.iconPiece(
        svg_source, center, scale=scale, piece_id=piece_id,
        fill=fill, outline=outline, outline_color=outline_color, attrs=attrs,
    )
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

tiers = [
    ('board', 0.5,  'solid fill'),
    ('list',  1.5,  'simple pattern'),
    ('large', 3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx, cy = 80 + i * spacing, 90
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    flag.draw_icon(fishPattern, MapCord(cx, cy), canvas,
                   scale=scale, size=size,
                   piece_id=f"bear_{size}", layer=f"bear_{i}",
                   outline=0.06)

    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" class="{lbl.name}">{size} (×{scale})</text>')

canvas.show()


I guess the issue is that is isn't a closed path. we still have a polar bear as opposed to a swirl one

@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board',
               scale: float = 1.0) -> tuple:
    if size == 'board':
        return self.primary, None

    if size == 'list':
        _SIMPLE = [
            self.bicolorH, self.bicolorV, self.tricolorH,
            self.quarters, self.crossFlag, self.chevron,
        ]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        pat = fn(pat_id)
        pat.attributes['patternTransform'] = f'scale({scale:.2f})'
        return f"url(#{pat_id})", pat

    # 'large' — normalize to ~40px visual tile size
    TARGET_TILE = 40.0
    pat = self.flagPattern(pat_id)  # no scale override yet
    native_w = float(pat.attributes.get('width', 100))
    # Read any existing scale from the pattern
    existing = pat.attributes.get('patternTransform', 'scale(1)')
    m = re.search(r'scale\(([\d.]+)\)', existing)
    base_scale = float(m.group(1)) if m else 1.0
    effective_tile = native_w * base_scale
    # Adjust so the visual tile is ~TARGET_TILE
    adjust = TARGET_TILE / effective_tile if effective_tile > 0 else 1.0
    pat.attributes['patternTransform'] = f'scale({base_scale * adjust:.4f})'
    return f"url(#{pat_id})", pat


@patch
def iconPiece(self: CountryFlag, svg_source: str, center: MapCord,
              scale: float = 1.0, piece_id: str = None,
              fill: str = None, outline: float = 0,
              outline_color: str = None, solid: bool = True,
              detail: bool = True, detail_color: str = None,
              attrs: dict = None):
    """Render an arbitrary SVG icon at center using a 3-layer approach:
    
    1. Outline layer (optional): slightly scaled-up solid behind for border
    2. Base layer: silhouette (first subpath only) — solid fill, no holes
    3. Detail layer (optional): full original paths on top, stroke-only,
       to restore interior line detail
    
    svg_source: raw SVG string — viewBox used for sizing.
    Normalized to ~45px baseline (chess piece compatible).
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}"
    if fill is None:
        fill = self.primary
    if outline_color is None:
        outline_color = self.comp
    if detail_color is None:
        detail_color = self.comp

    # Parse viewBox
    vb_match = re.search(r'viewBox=["\']([^"\']+)["\']', svg_source)
    if vb_match:
        parts = vb_match.group(1).split()
        vb_x, vb_y = float(parts[0]), float(parts[1])
        vb_w, vb_h = float(parts[2]), float(parts[3])
    else:
        vb_x, vb_y, vb_w, vb_h = 0, 0, 512, 512

    # Full inner content (cleaned)
    inner_full = re.sub(r'<svg[^>]*>', '', svg_source)
    inner_full = re.sub(r'</svg>\s*$', '', inner_full).strip()
    inner_full = re.sub(r'\s*fill="[^"]*"', '', inner_full)
    inner_full = re.sub(r'\s*stroke="[^"]*"', '', inner_full)

    # Silhouette: keep only first M...Z from each path
    def _first_subpath(m):
        d = m.group(1)
        z = d.find('Z')
        if z < 0: z = d.find('z')
        return f'd="{d[:z+1]}"' if z >= 0 else f'd="{d}"'
    inner_silhouette = re.sub(r'd="([^"]*)"', _first_subpath, inner_full)

    # Normalize to 45px baseline
    norm = 45.0 / max(vb_w, vb_h)
    cx_vb = vb_w / 2 + vb_x
    cy_vb = vb_h / 2 + vb_y
    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)

    solid_sw = max(vb_w, vb_h) * 0.15 if solid else 0
    stroke_attr = f'stroke="{fill}" stroke-width="{solid_sw:.1f}" stroke-linejoin="round"' if solid else 'stroke="none"'

    base_transform = f"translate({tx},{ty}) scale({scale * norm:.6f}) translate({-cx_vb:.1f},{-cy_vb:.1f})"

    layers = []

    # 1. Outline layer
    if outline > 0:
        border_s = 1.0 + outline
        border_transform = (
            f"translate({tx},{ty}) scale({scale * norm * border_s:.6f}) "
            f"translate({-cx_vb:.1f},{-cy_vb:.1f})"
        )
        outline_sw = f'stroke="{outline_color}" stroke-width="{solid_sw:.1f}" stroke-linejoin="round"' if solid else 'stroke="none"'
        layers.append(
            f'<g fill="{outline_color}" {outline_sw} '
            f'transform="{border_transform}">\n{inner_silhouette}\n</g>'
        )

    # 2. Base layer: solid silhouette (no holes)
    layers.append(
        f'<g fill="{fill}" {stroke_attr} '
        f'transform="{base_transform}">\n{inner_silhouette}\n</g>'
    )

    # 3. Detail layer: full paths, stroke-only on top
    if detail:
        detail_sw = max(vb_w, vb_h) * 0.005  # thin detail lines
        layers.append(
            f'<g fill="none" stroke="{detail_color}" stroke-width="{detail_sw:.1f}" '
            f'stroke-linejoin="round" stroke-linecap="round" '
            f'transform="{base_transform}">\n{inner_full}\n</g>'
        )

    return f'''<g id="{piece_id}" style="cursor:pointer"{extra}>
{''.join(layers)}
</g>'''


@patch
def draw_icon(self: CountryFlag, svg_source: str, center: MapCord,
              builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
              piece_id: str = None, layer: str = None,
              opacity: float = 1.0, outline: float = 0,
              outline_color: str = None, solid: bool = True,
              attrs: dict = None) -> str:
    """Convenience: render icon with flag-colored fill at the right tier.
    
    size: 'board' (solid), 'list' (simple pattern), 'large' (full pattern)
    outline: border as fraction of size (0.06 = 6%), fill-based.
    solid: True fills interior of outline-style SVGs (default True).
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}_{id(svg_source) % 9999}"

    pat_id = f"{piece_id}_pat"
    fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)

    if pat_def is not None:
        builder.add_definition(pat_def)

    svg = self.iconPiece(
        svg_source, center, scale=scale, piece_id=piece_id,
        fill=fill, outline=outline, outline_color=outline_color,
        solid=solid, attrs=attrs,
    )
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


what does the final demo look like

In [ ]:
with resources.files('HexMagic').joinpath('data/patterns/animals/Fish.svg').open() as f:
    fishPattern = f.read()
fishPattern

'<svg id="Outline" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512"><path d="M495.99,183.613a8,8,0,0,0-11.3-6.9l-86.74,39.427c-24.881,2.158-48.33-5.6-72.122-16.319l9.763-29.3a8,8,0,0,0-3-9.083C251.455,104.653,186.716,104,184,104a8,8,0,0,0-7.59,5.471l-17.616,52.857C73.623,188,20.268,248.117,17.979,250.731a8,8,0,0,0-.565,9.811c42.952,62.28,93.822,100.236,151.442,113.118,2.54,9.735,12.534,29.979,51.566,49.495a8,8,0,0,0,10.234-2.717l16-24a8.01,8.01,0,0,0-4.7-12.195,71.807,71.807,0,0,1-14.765-6.069c33.838-2.058,64.98-11.573,90.372-22.409L337.753,381a8,8,0,0,0,9.825,2.157c1.375-.687,33.975-17.225,58.822-50.355a8,8,0,0,0-1.6-11.2l-12.366-9.275c4.14-3.121,7.347-5.694,9.528-7.5L485.9,327.718a8,8,0,0,0,10.1-7.844c-.377-24-10.438-47.3-26.826-63.207C494.706,236.588,497.206,208.735,495.99,183.613ZM189.648,120.354c16.58,1.511,67.682,9.438,128.86,50.825l-7.263,21.793q-5.478-2.655-11.011-5.392c-38-18.711-77.2-37.991-123.052-29.821ZM101.393,325.815Q65.31,300.129,34.3,256.746a330.425,330.425,0,0


flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

tiers = [
    ('board', 0.5,  'solid fill'),
    ('list',  1.5,  'simple pattern'),
    ('large', 3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx, cy = 80 + i * spacing, 90
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    flag.draw_icon(fishPattern, MapCord(cx, cy), canvas,
                   scale=scale, size=size,
                   piece_id=f"bear_{size}", layer=f"bear_{i}",
                   outline=0.06, solid=True)

    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" class="{lbl.name}">{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Bear at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()


Ok. this is all a mess. I think we want this to have a primary stroke and have a circle background in comp. no patterns just colors



@patch
def draw_icon(self: CountryFlag, svg_source: str, center: MapCord,
              builder: SVGBuilder, scale: float = 1.0,
              piece_id: str = None, layer: str = None,
              opacity: float = 1.0, solid: bool = True,
              detail: bool = True, attrs: dict = None) -> str:
    """Render icon on builder. No patterns — just solid colors + circle bg."""
    if piece_id is None:
        piece_id = f"icon_{self.name}_{id(svg_source) % 9999}"

    svg = self.iconPiece(
        svg_source, center, scale=scale, piece_id=piece_id,
        solid=solid, detail=detail, attrs=attrs,
    )
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


In [ ]:
with resources.files('HexMagic').joinpath('data/patterns/animals/Crab.svg').open() as f:
    fishPattern = f.read()

flag = CountryFlag.seaborn("husl", 1)[0]

canvas = SVGBuilder()

scales = [0.5, 1.5, 3.5]
bg_pad = 1.3
max_r = max(s * 22.5 * bg_pad for s in scales)
margin = 30

canvas.height = int(max_r * 2 + margin * 2)
canvas.width = int(sum(s * 22.5 * bg_pad * 2 + 40 for s in scales) + margin * 2)

cx = margin
for i, scale in enumerate(scales):
    r = scale * 22.5 * bg_pad
    cx += r + 20
    cy = canvas.height // 2
    flag.draw_icon(fishPattern, MapCord(cx, cy), canvas,
                   scale=scale, piece_id=f"fish_{i}", layer=f"fish_{i}")
    cx += r + 20

canvas.show()


I think I want a much ligther comp for the color. what colors would you pick?

can you write the function with the new colors?

In [ ]:
!ls ../HexMagic/data/patterns/animals/*.svg

ls: cannot access '../HexMagic/data/patterns/animals/*.svg': No such file or directory


can you uses thse to create random animals and have a patch function on CountryFlag which returns a svg (in three sizes)

In [ ]:
## Master

In [ ]:
#| export
@patch
def iconPiece(self: CountryFlag, svg_source: str, center: MapCord,
              scale: float = 1.0, piece_id: str = None,
              fill: str = None, bg_color: str = None,
              bg_padding: float = 1.3, solid: bool = True,
              detail: bool = True, detail_color: str = None,
              attrs: dict = None):
    """Render an SVG icon as a colored silhouette on a circle background.
    
    svg_source: raw SVG string — viewBox used for sizing.
    Normalized to ~45px baseline (chess piece compatible).
    
    fill:       silhouette color (default: flag.primary)
    bg_color:   circle background (default: flag.lightComp)
    bg_padding: circle radius as multiple of icon half-size (1.3 = 30% padding)
    solid:      True = thick self-stroke to fill outline-style SVGs
    detail:     True = draw thin detail lines on top
    """
    if piece_id is None:
        piece_id = f"icon_{self.name}"
    if fill is None:
        fill = self.primary
    if bg_color is None:
        bg_color = self.lightComp
    if detail_color is None:
        detail_color = self.darkPrimary

    # Parse viewBox
    vb_match = re.search(r'viewBox=["\']([^"\']+)["\']', svg_source)
    if vb_match:
        parts = vb_match.group(1).split()
        vb_x, vb_y = float(parts[0]), float(parts[1])
        vb_w, vb_h = float(parts[2]), float(parts[3])
    else:
        vb_x, vb_y, vb_w, vb_h = 0, 0, 512, 512

    # Clean inner content
    inner_full = re.sub(r'<svg[^>]*>', '', svg_source)
    inner_full = re.sub(r'</svg>\s*$', '', inner_full).strip()
    inner_full = re.sub(r'\s*fill="[^"]*"', '', inner_full)
    inner_full = re.sub(r'\s*stroke="[^"]*"', '', inner_full)

    # Silhouette: first subpath only (outer contour)
    def _first_subpath(m):
        d = m.group(1)
        z = d.find('Z')
        if z < 0: z = d.find('z')
        return f'd="{d[:z+1]}"' if z >= 0 else f'd="{d}"'
    inner_silhouette = re.sub(r'd="([^"]*)"', _first_subpath, inner_full)

    # Normalize to 45px baseline
    norm = 45.0 / max(vb_w, vb_h)
    cx_vb = vb_w / 2 + vb_x
    cy_vb = vb_h / 2 + vb_y
    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)

    solid_sw = max(vb_w, vb_h) * 0.15 if solid else 0
    stroke_attr = f'stroke="{fill}" stroke-width="{solid_sw:.1f}" stroke-linejoin="round"' if solid else 'stroke="none"'

    base_transform = f"translate({tx},{ty}) scale({scale * norm:.6f}) translate({-cx_vb:.1f},{-cy_vb:.1f})"

    # Circle radius in final px
    circle_r = 22.5 * scale * bg_padding

    layers = []

    # 1. Circle background
    layers.append(
        f'<circle cx="{tx}" cy="{ty}" r="{circle_r:.1f}" '
        f'fill="{bg_color}" stroke="none"/>'
    )

    # 2. Solid silhouette
    layers.append(
        f'<g fill="{fill}" {stroke_attr} '
        f'transform="{base_transform}">\n{inner_silhouette}\n</g>'
    )

    # 3. Detail lines on top
    if detail:
        detail_sw = max(vb_w, vb_h) * (0.003 + 0.002 * scale)
        layers.append(
            f'<g fill="none" stroke="{detail_color}" stroke-width="{detail_sw:.1f}" '
            f'stroke-linejoin="round" stroke-linecap="round" '
            f'transform="{base_transform}">\n{inner_full}\n</g>'
        )

    return f'''<g id="{piece_id}" style="cursor:pointer"{extra}>
{''.join(layers)}
</g>'''


In [ ]:
#| export
from pathlib import Path

# Load all animal SVGs into a dict at import time
_ANIMAL_SVGS = {}
_ANIMAL_NAMES = []

_animal_dir = resources.files('HexMagic').joinpath('data/patterns/animals')
for f in sorted(_animal_dir.iterdir()):
    if f.name.endswith('.svg'):
        name = f.name.removesuffix('.svg')
        with f.open() as fh:
            _ANIMAL_SVGS[name] = fh.read()
        _ANIMAL_NAMES.append(name)

#print(f"Loaded {len(_ANIMAL_NAMES)} animals: {_ANIMAL_NAMES[:5]}...")


In [ ]:
#| export
@patch
def animal_svg(self: CountryFlag, size: str = 'board',
               animal: str = None, center: MapCord = None,
               piece_id: str = None, detail: bool = True,
               attrs: dict = None) -> str:
    """Return SVG string for this flag's animal icon.
    
    Args:
        size:   'board' (small, scale=0.5), 'list' (medium, scale=1.5), 
                'large' (big, scale=3.5)
        animal: animal name (e.g. 'Bear'). If None, picks deterministically
                from flag name hash so it's stable across calls.
        center: placement position (default: centered for the size)
        piece_id: SVG element id
        detail: draw interior detail lines
        attrs:  dict of HTMX/HTML attributes
    
    Returns: raw SVG string (caller adds to builder or wraps in <svg>)
    """
    _SIZES = {'board': 0.5, 'list': 1.5, 'large': 3.5}
    scale = _SIZES.get(size, 1.0)
    
    # Deterministic animal from name if not specified
    if animal is None:
        idx = hash(self.name) % len(_ANIMAL_NAMES)
        animal = _ANIMAL_NAMES[idx]
    
    svg_source = _ANIMAL_SVGS.get(animal)
    if svg_source is None:
        raise ValueError(f"Unknown animal '{animal}'. Available: {_ANIMAL_NAMES}")
    
    # Default center: sized to fit
    if center is None:
        r = 22.5 * scale * 1.3
        center = MapCord(r + 4, r + 4)
    
    if piece_id is None:
        piece_id = f"animal_{self.name}_{size}"
    
    return self.iconPiece(
        svg_source, center, scale=scale, piece_id=piece_id,
        detail=detail, attrs=attrs,
    )


@patch
def animal_name(self: CountryFlag) -> str:
    """Return the deterministic animal name for this flag."""
    return _ANIMAL_NAMES[hash(self.name) % len(_ANIMAL_NAMES)]


In [ ]:
flags = CountryFlag.seaborn("husl", 5)
canvas = SVGBuilder()
canvas.add_font('Cinzel')

sizes = ['board', 'list', 'large']
scales = {'board': 0.5, 'list': 1.5, 'large': 3.5}
col_widths = [80, 120, 220]  # px per column
row_height = 230
margin = 20

canvas.width = sum(col_widths) + margin * 4
canvas.height = len(flags) * row_height + margin

for row, flag in enumerate(flags):
    animal = flag.animal_name()
    cx_offset = margin
    cy = margin + row * row_height + 110
    
    for col, size in enumerate(sizes):
        scale = scales[size]
        r = 22.5 * scale * 1.3
        cx = cx_offset + col_widths[col] // 2
        
        svg = flag.animal_svg(size=size, center=MapCord(cx, cy),
                              piece_id=f"a_{row}_{col}")
        canvas.adjust(f"animal_{row}_{col}", svg)
        cx_offset += col_widths[col] + margin
    
    # Label
    lbl = flag.labelStyle(f"lbl_{row}")
    canvas.add_style(lbl)
    canvas.adjust(f"name_{row}",
        f'<text x="{canvas.width - 10}" y="{cy}" text-anchor="end" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{flag.name} — {animal}</text>')

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_Wy5BhjhESBmgjpbbmV19-g&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;500&quot; height=&quot;1170&quot; viewBox=&quot;0 0 500 1170&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;style&gt;
@import url(&#x27;https://fonts.googleapis.com/css2?family=Cinzel&amp;display=swap&#x27;);
.contrast_lbl_4 {
  fill:#472b55;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_3 {
  fill:#143b45;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_2 {
  fill:#123e2b;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_1 {
  fill:#3d3711;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_0 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;animal_0_0&quot;&gt;
&lt;g id=&quot;a_0_0&quot; style=&quot;cursor:pointer&quot;&gt;
&lt;circle cx=&quot;60&quot; cy=&quot;130&quot; r=&quot;14.6&quot; fill=&quot;#def2ef&quot; stroke=&quot;none&quot;/&gt;&lt;g fill=&quot;#ffc8d1&quot; stroke=&quot;#ffc8d1&quot; stroke-width=&quot;76.8&quot; stroke-linejoin=&quot;round&quot; transform=&quot;translate(60,130) scale(0.043945) translate(-256.0,-256.0)&quot;&gt;
&lt;path d=&quot;M168,136a8,8,0,1,1,8,8A8,8,0,0,1,168,136Z&quot;/&gt;
&lt;/g&gt;&lt;g fill=&quot;none&quot; stroke=&quot;#562730&quot; stroke-width=&quot;2.0&quot; stroke-linejoin=&quot;round&quot; stroke-linecap=&quot;round&quot; transform=&quot;translate(60,130) scale(0.043945) translate(-256.0,-256.0)&quot;&gt;
&lt;path d=&quot;M168,136a8,8,0,1,1,8,8A8,8,0,0,1,168,136ZM128,300.888V287.877c-39.192-30.085-68.387-70.943-86.8-121.515-.163-.424-.314-.843-.468-1.266l-.126-.35a40.041,40.041,0,0,1,28.7-52.614c14.819-3.391,37.382-3.692,63.615,12.251a78.958,78.958,0,0,1,9.1-12.7C154.594,97.54,169.775,92.2,185.94,96.239c26.111,6.528,35.559,27.707,37.607,41.844l133.183,44.4a23.893,23.893,0,0,1,13.249,10.867l24.642,43.12c16.754-3.059,29.06-4.18,31.829-4.407,4.718-.852,38.055-6,57.092,14.4,13.42,14.377,16.005,37.136,7.685,67.643a8.01,8.01,0,0,1-1.8,3.278l-60.956,67.051A72.145,72.145,0,0,1,375.2,408H219.509a7.977,7.977,0,0,1-1.75-.2A8.03,8.03,0,0,1,216,408H192a8,8,0,0,1,0-16h24a8.053,8.053,0,0,1,1.467.1

In [ ]:
#| export
@patch
def country_animals(self: CountryFlag, n: int = 6) -> list:
    """Pick n distinct animals for this country, deterministic by name.
    Returns list of animal name strings."""
    rng = random.Random(hash(self.name))
    return rng.sample(_ANIMAL_NAMES, min(n, len(_ANIMAL_NAMES)))


In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)
animals = flag.country_animals(8)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

cols = 4
rows = 2
spacing_x = 130
spacing_y = 140
margin = 30
scale = 1.5

canvas.width = cols * spacing_x + margin * 2
canvas.height = rows * spacing_y + margin * 2 + 30  # extra for title

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="{margin}" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name}\'s {flag.countryPrefix} — 8 Patrols</text>')

for i, animal in enumerate(animals):
    row = i // cols
    col = i % cols
    cx = margin + col * spacing_x + spacing_x // 2
    cy = margin + 30 + row * spacing_y + 60

    svg = flag.animal_svg(size='list', animal=animal,
                          center=MapCord(cx, cy),
                          piece_id=f"patrol_{i}")
    canvas.adjust(f"patrol_{i}", svg)

    # Squad label
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + 52}" text-anchor="middle" '
        f'font-size="11" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{animal}s</text>'
        f'<text x="{cx}" y="{cy + 66}" text-anchor="middle" '
        f'font-size="9" font-family="sans-serif" fill="#888">'
        f'Patrol {i+1}</text>')

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_lCEIWyvVR4u8TiJg-_KaaQ&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;580&quot; height=&quot;370&quot; viewBox=&quot;0 0 580 370&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;style&gt;
@import url(&#x27;https://fonts.googleapis.com/css2?family=Cinzel&amp;display=swap&#x27;);
.contrast_lbl_7 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_6 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_5 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_4 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_3 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_2 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_1 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_0 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;title&quot;&gt;
&lt;text x=&quot;290&quot; y=&quot;30&quot; text-anchor=&quot;middle&quot; font-size=&quot;16&quot; font-family=&quot;&#x27;Cinzel&#x27;, sans-serif&quot; fill=&quot;#562730&quot;&gt;James&#x27;s Jurisdiction — 8 Patrols&lt;/text&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;patrol_0&quot;&gt;
&lt;g id=&quot;patrol_0&quot; style=&quot;cursor:pointer&quot;&gt;
&lt;circle cx=&quot;95&quot; cy=&quot;120&quot; r=&quot;43.9&quot; fill=&quot;#def2ef&quot; stroke=&quot;none&quot;/&gt;&lt;g fill=&quot;#ffc8d1&quot; stroke=&quot;#ffc8d1&quot; stroke-width=&quot;76.8&quot; stroke-linejoin=&quot;round&quot; transform=&quot;translate(95,120) scale(0.131836) translate(-256.0,-256.0)&quot;&gt;
&lt;path d=&quot;M98.343,138.343a54.972,54.972,0,0,0-5,5.758,48.063,48.063,0,0,1-5.761-10.66,8,8,0,0,0-15.17,5.087,65.526,65.526,0,0,0,12.993,21.2,55.5,55.5,0,0,0-2.009,19.694c.881,12.977,5.8,27.6,14.613,43.468,11.973,21.552,30.927,45.363,56.336,70.772,21.607,21.607,43.406,26.59,57.188,27.173-6.31,23.134.032,54.5,3.279,67.682l-14.6,9.733c-11.7-8.16-41.114-26.04-61.691-15.975-.83.406-1.627.855-2.4,1.334a56.15,56.15,0,0,0-26.06.635,8,8,0,1,0,3.884,15.522,44.648,44.648,0,0,1,10.254-1.369,62.348,62.348,0,0,0-4.049,

In [ ]:
#| export
SQUAD_DESCRIPTORS = {
    'A': ['Angry', 'Arcane', 'Armored', 'Ashen', 'Audacious', 'Awesome', 'Awakened', 'Azure'],
    'B': ['Bold', 'Brave', 'Brash', 'Blazing', 'Brutal', 'Bronze', 'Battle', 'Blood'],
    'C': ['Crimson', 'Cunning', 'Crazed', 'Charging', 'Cursed', 'Colossal', 'Cruel', 'Crowned'],
    'D': ['Dark', 'Deadly', 'Dire', 'Dread', 'Doomed', 'Daring', 'Defiant', 'Doom'],
    'E': ['Elite', 'Ember', 'Enraged', 'Eternal', 'Evil', 'Exiled', 'Ebon', 'Eerie'],
    'F': ['Fierce', 'Flaming', 'Furious', 'Feral', 'Fearless', 'Frozen', 'Fatal', 'Frenzied'],
    'G': ['Grim', 'Golden', 'Gallant', 'Ghostly', 'Grand', 'Grizzled', 'Gutsy', 'Glorious'],
    'H': ['Howling', 'Hellfire', 'Hardened', 'Haunted', 'Horned', 'Heavy', 'Hex', 'Hulking'],
    'I': ['Iron', 'Infernal', 'Icy', 'Immortal', 'Imperial', 'Indomitable', 'Ivory', 'Ill'],
    'J': ['Jade', 'Jagged', 'Jolly', 'Jaded', 'Jawbone', 'Jinxed', 'Juiced', 'Jungle'],
    'K': ['Keen', 'Killer', 'Kingsguard', 'Knighted', 'Kraken', 'Kindle', 'Kold', 'Karmic'],
    'L': ['Lone', 'Lunar', 'Lucky', 'Lethal', 'Lightning', 'Lurking', 'Lava', 'Legendary'],
    'M': ['Mighty', 'Mad', 'Merciless', 'Mystic', 'Marauding', 'Molten', 'Monstrous', 'Moon'],
    'N': ['Night', 'Noble', 'Nefarious', 'Nimble', 'Northern', 'Nuclear', 'Noxious', 'Notorious'],
    'O': ['Obsidian', 'Ominous', 'Outlaw', 'Onyx', 'Ornery', 'Old', 'Onslaught', 'Outraged'],
    'P': ['Primal', 'Phantom', 'Proud', 'Plated', 'Perilous', 'Poison', 'Pyro', 'Prowling'],
    'Q': ['Quick', 'Quarrel', 'Quaking', 'Quartz', 'Quiet', 'Quintuple', 'Questing', 'Quill'],
    'R': ['Raging', 'Royal', 'Reckless', 'Rusted', 'Roaring', 'Ravenous', 'Relentless', 'Red'],
    'S': ['Savage', 'Shadow', 'Storm', 'Steel', 'Scarlet', 'Sinister', 'Swift', 'Steadfast'],
    'T': ['Thunder', 'Twisted', 'Titanium', 'Terrible', 'Thorned', 'Toxic', 'Twilight', 'Tribal'],
    'U': ['Undying', 'Unholy', 'Unbroken', 'Ultra', 'Umber', 'Unleashed', 'Unseen', 'Unyielding'],
    'V': ['Venom', 'Vicious', 'Valor', 'Volatile', 'Void', 'Violent', 'Vigilant', 'Volcanic'],
    'W': ['War', 'Wicked', 'Wild', 'Wraith', 'Winter', 'Wrath', 'Wailing', 'Watchful'],
    'X': ['Xenon', 'Xeric', 'Xtreme', 'Xeno', 'Xerus', 'Xeric', 'Xanthic', 'Xiphos'],
    'Y': ['Yelling', 'Yellow', 'Yonder', 'Young', 'Yearning', 'Yanked', 'Yew', 'Yelping'],
    'Z': ['Zealous', 'Zero', 'Zodiac', 'Zephyr', 'Zinc', 'Zoning', 'Zenith', 'Zombie'],
}

@patch
def squad_name(self: CountryFlag, animal: str, seed: int = 0) -> str:
    """Generate an alliterative squad name like 'Fierce Foxes'.
    
    Uses first letter of animal to pick a matching descriptor.
    seed: varies the descriptor (0-7) for multiple squads with same animal letter.
    """
    letter = animal[0].upper()
    descriptors = SQUAD_DESCRIPTORS.get(letter, ['Grand'])
    rng = random.Random(hash((self.name, animal, seed)))
    descriptor = rng.choice(descriptors)
    # Simple pluralization
    if animal.endswith(('s', 'sh', 'ch', 'x')):
        plural = animal + 'es'
    elif animal.endswith('y') and animal[-2] not in 'aeiou':
        plural = animal[:-1] + 'ies'
    else:
        plural = animal + 's'
    return f"{descriptor} {plural}"

@patch
def country_squads(self: CountryFlag, n: int = 6) -> list:
    """Return list of (animal, squad_name) tuples for this country's squads."""
    animals = self.country_animals(n)
    return [(a, self.squad_name(a, i)) for i, a in enumerate(animals)]


In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
for animal, name in flag.country_squads(8):
    print(f"  {name:30s} ({animal})")


  Roaring Rabbits                (Rabbit)
  Bronze Beetles                 (Beetle)
  Brash Bats                     (Bat)
  Hex Horses                     (Horse)
  Cruel Cats                     (Cat)
  Outraged Ostriches             (Ostrich)
  Cunning Cows                   (Cow)
  Battle Bears                   (Bear)


In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
squads = flag.country_squads(8)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

cols, rows = 4, 2
spacing_x, spacing_y = 140, 140
margin = 20

canvas.width = cols * spacing_x + margin * 2
canvas.height = rows * spacing_y + margin * 2 + 30

canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="{margin + 4}" text-anchor="middle" '
    f'font-size="15" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name}\'s {flag.countryPrefix} — Squads</text>')

for i, (animal, squad) in enumerate(squads):
    row, col = divmod(i, cols)
    cx = margin + col * spacing_x + spacing_x // 2
    cy = margin + 34 + row * spacing_y + 55

    svg = flag.animal_svg(size='list', animal=animal,
                          center=MapCord(cx, cy),
                          piece_id=f"sq_{i}")
    canvas.adjust(f"sq_{i}", svg)

    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"lbl_{i}",
        f'<text x="{cx}" y="{cy + 52}" text-anchor="middle" '
        f'font-size="10" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{squad}</text>')

canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_1aJAqzNBTeGXcmNyCVSfLQ&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;600&quot; height=&quot;350&quot; viewBox=&quot;0 0 600 350&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;style&gt;
@import url(&#x27;https://fonts.googleapis.com/css2?family=Cinzel&amp;display=swap&#x27;);
.contrast_lbl_7 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_6 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_5 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_4 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_3 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_2 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_1 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
.contrast_lbl_0 {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;title&quot;&gt;
&lt;text x=&quot;300&quot; y=&quot;24&quot; text-anchor=&quot;middle&quot; font-size=&quot;15&quot; font-family=&quot;&#x27;Cinzel&#x27;, sans-serif&quot; fill=&quot;#562730&quot;&gt;Mary&#x27;s Mount — Squads&lt;/text&gt;
&lt;/g&gt;
&lt;g data-layer=&quot;sq_0&quot;&gt;
&lt;g id=&quot;sq_0&quot; style=&quot;cursor:pointer&quot;&gt;
&lt;circle cx=&quot;90&quot; cy=&quot;109&quot; r=&quot;43.9&quot; fill=&quot;#def2ef&quot; stroke=&quot;none&quot;/&gt;&lt;g fill=&quot;#ffc8d1&quot; stroke=&quot;#ffc8d1&quot; stroke-width=&quot;76.8&quot; stroke-linejoin=&quot;round&quot; transform=&quot;translate(90,109) scale(0.131836) translate(-256.0,-256.0)&quot;&gt;
&lt;path d=&quot;M440,144a8,8,0,1,1-8-8A8,8,0,0,1,440,144Z&quot;/&gt;
&lt;/g&gt;&lt;g fill=&quot;none&quot; stroke=&quot;#562730&quot; stroke-width=&quot;3.1&quot; stroke-linejoin=&quot;round&quot; stroke-linecap=&quot;round&quot; transform=&quot;translate(90,109) scale(0.131836) translate(-256.0,-256.0)&quot;&gt;
&lt;path d=&quot;M440,144a8,8,0,1,1-8-8A8,8,0,0,1,440,144Zm33.9-25.551C484.123,130.542,496,150.154,496,176c0,13.22-3.958,22.335-11.766,27.091a24.635,24.635,0,0,1-13.043,3.365c-7.595,0-15.758-2.573-23.211-5.726a39.836,39

### Glyphs

In [ ]:
!cat ../../HexMagic/styles.py

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/01_styles.ipynb.

# %% auto #0
__all__ = ['NamedColor', 'EASING_PRESETS', 'daisy_hdrs', 'app', 'rt', 'preview', 'simpleSVG', 'StyleDemo', 'tag', 'indent',
           'Generatable', 'SVGDef', 'StyleCSS', 'LoopingLayerAnimation', 'apply_looping_animation', 'LayerAnimation',
           'SVGLayer', 'SVGBuilder', 'get_preview', 'SVGPatternLoader']

# %% ../nbs/01_styles.ipynb #574e449f
import numpy as np

from dataclasses import dataclass
from collections import namedtuple
from typing import Optional, Literal

from fastcore.basics import patch
from collections import namedtuple

#For parsing patterns
import xml.etree.ElementTree as ET
from abc import ABC, abstractmethod
from bs4 import BeautifulSoup
import lxml

# fun with colors
import colorsys
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, to_hex


from IPython.display import SVG, HTML

# %% ../nbs/01_styles.ipynb #8793c26f
import os
import has

In [ ]:
!cat ../../HexMagic/plot/*.py

# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/plots/02e_HexChunk.ipynb.

# %% auto #0
__all__ = ['ChunkWorldMap', 'ChunkLocalMap', 'HexChunk']

# %% ../../nbs/plots/02e_HexChunk.ipynb #0320a5bf
import sys
import math
import numpy as np
import math
from collections import namedtuple
from dataclasses import dataclass, field
from fastcore.basics import patch
from dataclasses import dataclass, field
from typing import Iterator
import numpy as np


# %% ../../nbs/plots/02e_HexChunk.ipynb #4380d4d6
from .primitives import MapCord, MapSize, MapRect, MapPath, PrimitiveDemo
from .cube import HexPosition
from .hex import Hex, HexGrid

from ..styles import StyleCSS,  SVGBuilder
from typing import NamedTuple

# %% ../../nbs/plots/02e_HexChunk.ipynb #d7a56ebd
class ChunkWorldMap(NamedTuple):
    idx: int
    local_pos: HexPosition
    world_pos: HexPosition

class ChunkLocalMap(NamedTuple):
    idx: int
    pos: HexPosition
    

# %% ../../nbs/plots/02e_HexChunk.ipynb #04a49083
class HexCh

In [ ]:
#| export
from dataclasses import dataclass

In [ ]:
#| export

import math

import numpy as np

In [ ]:
#| export
@dataclass
class DiagramGlyphs:
    """SVG glyph renderer for piece-plan overlays, themed from a CountryFlag."""
    flag:         CountryFlag
    size:         float = 10.0
    stroke_width: float = 1.5
    opacity:      float = 0.85

    @property
    def color(self):    return self.flag.darkPrimary
    @property
    def accent(self):   return self.flag.comp
    @property
    def fill(self):     return self.flag.primary
    @property
    def light(self):    return self.flag.lightPrimary
    @property
    def danger(self):   return '#c0392b'
    @property
    def _n(self):       return self.flag.name

    # ── Style factories ─────────────────────────────────────
    def outline_style(self) -> StyleCSS:
        """Stroke only — circles, arcs, rings."""
        return StyleCSS(f"glyph_{self._n}",
            stroke=self.color, fill="none",
            stroke_width=self.stroke_width, opacity=self.opacity,
            stroke_linecap="round")

    def filled_style(self) -> StyleCSS:
        """Stroke + primary fill — shields, houses, gifts."""
        return StyleCSS(f"glyph_fill_{self._n}",
            stroke=self.color, fill=self.fill,
            stroke_width=self.stroke_width, opacity=self.opacity)

    def accent_style(self) -> StyleCSS:
        """Accent color — highlights & decorations."""
        return StyleCSS(f"glyph_accent_{self._n}",
            stroke=self.accent, fill=self.accent, opacity=self.opacity)

    def solid_style(self) -> StyleCSS:
        """Solid dark fill, no stroke — arrowheads."""
        return StyleCSS(f"glyph_solid_{self._n}",
            stroke="none", fill=self.color, opacity=self.opacity)

    def stalk_style(self) -> StyleCSS:
        """Thick round-cap strokes — sheaf stalks."""
        return StyleCSS(f"glyph_stalk_{self._n}",
            stroke=self.color, fill="none",
            stroke_width=max(1.5, self.size * 0.15),
            stroke_linecap="round", opacity=self.opacity)

    def band_style(self) -> StyleCSS:
        """Thinner strokes — sheaf cross-bands."""
        return StyleCSS(f"glyph_band_{self._n}",
            stroke=self.color, fill="none",
            stroke_width=max(1, self.size * 0.1), opacity=self.opacity)


@patch
def register_styles(self: DiagramGlyphs, builder: SVGBuilder):
    """Register glyph CSS classes on builder for class-based styling."""
    builder.add_style(StyleCSS(f"glyph_{self.flag.name}", 
        stroke=self.color, fill="none", stroke_width=self.stroke_width, opacity=self.opacity))
    builder.add_style(StyleCSS(f"glyph_fill_{self.flag.name}",
        stroke=self.color, fill=self.fill, stroke_width=self.stroke_width, opacity=self.opacity))
    builder.add_style(StyleCSS(f"glyph_accent_{self.flag.name}",
        stroke=self.accent, fill=self.accent, opacity=self.opacity))


In [ ]:
#| export
@patch
def register_styles(self: DiagramGlyphs, builder: SVGBuilder):
    """Register all glyph CSS classes on an SVGBuilder."""
    for fn in (self.outline_style, self.filled_style, self.accent_style,
               self.solid_style, self.stalk_style, self.band_style):
        builder.add_style(fn())


@patch
def circle(self: DiagramGlyphs, center: MapCord, r: float,
           style: StyleCSS = None) -> str:
    s = style or self.outline_style()
    return f'<circle cx="{center.x:.1f}" cy="{center.y:.1f}" r="{r:.1f}" class="{s.name}"/>\n'


@patch
def ring(self: DiagramGlyphs, center: MapCord, r: float,
         style: StyleCSS = None) -> str:
    return self.circle(center, r, style or self.outline_style())


@patch
def arc(self: DiagramGlyphs, center: MapCord, r: float,
        start_deg: float, end_deg: float,
        style: StyleCSS = None) -> str:
    s = style or self.outline_style()
    sa, ea = math.radians(start_deg), math.radians(end_deg)
    p1 = MapCord(center.x + r * math.cos(sa), center.y + r * math.sin(sa))
    p2 = MapCord(center.x + r * math.cos(ea), center.y + r * math.sin(ea))
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {p1.x:.1f},{p1.y:.1f} A {r:.1f},{r:.1f} 0 {large} 1 '
            f'{p2.x:.1f},{p2.y:.1f}" class="{s.name}"/>\n')


@patch
def shield(self: DiagramGlyphs, center: MapCord, size: float = None,
           style: StyleCSS = None) -> str:
    """Shield/badge shape with curved bottom via polygon approximation."""
    sz = size or self.size
    style = style or self.filled_style()
    w, h = sz, sz * 1.2
    points = [
        MapCord(center.x - w*0.5, center.y - h*0.5),
        MapCord(center.x + w*0.5, center.y - h*0.5),
        MapCord(center.x + w*0.5, center.y + h*0.1),
        MapCord(center.x + w*0.3, center.y + h*0.3),
        MapCord(center.x,         center.y + h*0.5),
        MapCord(center.x - w*0.3, center.y + h*0.3),
        MapCord(center.x - w*0.5, center.y + h*0.1),
    ]
    return MapPath(points, style).drawClosed()


@patch
def sheaf(self: DiagramGlyphs, center: MapCord, size: float = None) -> str:
    """Wheat sheaf — three stalks with a horizontal band."""
    s = size or self.size
    svg = ""
    for dx in [-s*0.3, 0, s*0.3]:
        svg += MapPath([
            MapCord(center.x + dx,     center.y - s*0.5),
            MapCord(center.x + dx*0.3, center.y + s*0.5),
        ], self.stalk_style()).drawPolygon()
    svg += MapPath([
        MapCord(center.x - s*0.35, center.y),
        MapCord(center.x + s*0.35, center.y),
    ], self.band_style()).drawPolygon()
    return svg


@patch
def gift(self: DiagramGlyphs, center: MapCord, size: float = None,
         style: StyleCSS = None) -> str:
    s = (size or self.size) * 0.8
    style = style or self.filled_style()
    # Rounded box (rect needed for rx)
    svg = (f'<rect x="{center.x - s*0.5:.1f}" y="{center.y - s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'rx="{s*0.1:.1f}" class="{style.name}"/>\n')
    # Ribbon cross
    svg += MapPath([
        MapCord(center.x, center.y - s*0.4),
        MapCord(center.x, center.y + s*0.4),
    ], self.outline_style()).drawPolygon()
    svg += MapPath([
        MapCord(center.x - s*0.5, center.y),
        MapCord(center.x + s*0.5, center.y),
    ], self.outline_style()).drawPolygon()
    return svg


@patch
def house(self: DiagramGlyphs, center: MapCord, size: float = None,
          style: StyleCSS = None) -> str:
    """Combined roof + body as a single closed polygon."""
    s = size or self.size
    style = style or self.filled_style()
    points = [
        MapCord(center.x,         center.y - s*0.45),   # roof peak
        MapCord(center.x + s*0.5, center.y),             # right eave
        MapCord(center.x + s*0.4, center.y),             # right wall top
        MapCord(center.x + s*0.4, center.y + s*0.5),     # right wall bottom
        MapCord(center.x - s*0.4, center.y + s*0.5),     # left wall bottom
        MapCord(center.x - s*0.4, center.y),             # left wall top
        MapCord(center.x - s*0.5, center.y),             # left eave
    ]
    return MapPath(points, style).drawClosed()


@patch
def pause(self: DiagramGlyphs, center: MapCord, size: float = None,
          style: StyleCSS = None) -> str:
    s = size or self.size
    style = style or self.filled_style()
    w, h = s * 0.2, s * 0.6
    svg = ""
    for dx in [-s*0.18, s*0.18]:
        svg += (f'<rect x="{center.x + dx - w/2:.1f}" y="{center.y - h/2:.1f}" '
                f'width="{w:.1f}" height="{h:.1f}" '
                f'rx="{w*0.2:.1f}" class="{style.name}"/>\n')
    return svg


@patch
def rot_arrow(self: DiagramGlyphs, center: MapCord, r: float = None,
              clockwise: bool = True, style: StyleCSS = None) -> str:
    """Curved rotation arrow with filled arrowhead."""
    r = r or self.size * 0.5
    style = style or self.outline_style()
    if clockwise:
        arc_svg = self.arc(center, r, -120, 150, style)
        tip = MapCord(center.x + r * math.cos(math.radians(150)),
                      center.y + r * math.sin(math.radians(150)))
        ah = [tip,
              MapCord(tip.x - r*0.25, tip.y - r*0.15),
              MapCord(tip.x - r*0.1,  tip.y + r*0.2)]
    else:
        arc_svg = self.arc(center, r, 30, 300, style)
        tip = MapCord(center.x + r * math.cos(math.radians(30)),
                      center.y + r * math.sin(math.radians(30)))
        ah = [tip,
              MapCord(tip.x + r*0.25, tip.y - r*0.15),
              MapCord(tip.x + r*0.1,  tip.y + r*0.2)]
    return arc_svg + MapPath(ah, self.solid_style()).drawClosed()


I am refactoring DiagramGlyphs so that it uses StyleCSS, CountryFlag, and existing primitives like MapCord and MapPath as much as possible. can you rewrite these functions using them?

In [ ]:
#| export
@patch
def danger_style(self: DiagramGlyphs) -> StyleCSS:
    """Bold red strokes — death markers, warnings."""
    return StyleCSS(f"glyph_danger_{self._n}",
        stroke=self.danger, fill="none",
        stroke_width=max(3.0, self.size * 0.12),
        stroke_linecap="round", opacity=0.85)


@patch
def register_styles(self: DiagramGlyphs, builder: SVGBuilder):
    """Register all glyph CSS classes on an SVGBuilder."""
    for fn in (self.outline_style, self.filled_style, self.accent_style,
               self.solid_style, self.stalk_style, self.band_style,
               self.danger_style):
        builder.add_style(fn())


@patch
def food_pip(self: DiagramGlyphs, center: MapCord, frac: float,
             max_r=None, min_r=None, opacity=None) -> str:
    """Filled circle sized by food fraction (0–1), colored red→green."""
    max_r   = max_r or self.size * 0.7
    min_r   = min_r or self.size * 0.12
    opacity = opacity if opacity is not None else self.opacity
    f       = np.clip(frac, 0, 1)
    r       = min_r + (max_r - min_r) * f
    color   = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
    # Dynamic per-fraction color — one-off style (not pre-registered)
    pip_style = StyleCSS(f"pip_{int(f*100)}",
        fill=color, stroke=color, stroke_width=0.8, opacity=opacity)
    return self.circle(center, r, pip_style)


@patch
def death_marker(self: DiagramGlyphs, center: MapCord, size=None,
                 style: StyleCSS = None) -> str:
    """Bold X cross marking a death location."""
    s     = size or self.size * 0.35
    style = style or self.danger_style()
    svg   = ''
    for dx1, dy1, dx2, dy2 in [(-s, -s, s, s), (s, -s, -s, s)]:
        svg += MapPath([
            MapCord(center.x + dx1, center.y + dy1),
            MapCord(center.x + dx2, center.y + dy2),
        ], style).drawPolygon()
    return svg


@patch
def transfer_arrow(self: DiagramGlyphs, start: MapCord, end: MapCord,
                   style: StyleCSS = None, arrow_size: float = None) -> str:
    """Directed arrow line for food/resource transfers."""
    style      = style or self.outline_style()
    arrow_size = arrow_size or max(4.0, self.size * 0.3)
    return MapPath([start, end], style).with_arrowhead(arrow_size=arrow_size)


I am refactoring DiagramGlyphs so that it uses StyleCSS, CountryFlag, and existing primitives like MapCord and MapPath as much as possible. You are doing a great job. can you rewrite these functions ( food_pip, death_marker, transfer_arrow) using them?

In [ ]:
#| export
@patch
def blocked_fill_style(self: DiagramGlyphs) -> StyleCSS:
    """Red filled circle for blocked markers."""
    return StyleCSS(f"glyph_blocked_{self._n}",
        fill="red", stroke="darkred", stroke_width=1, opacity=0.55)

@patch
def blocked_cross_style(self: DiagramGlyphs) -> StyleCSS:
    """White cross strokes inside blocked markers."""
    return StyleCSS(f"glyph_blocked_x_{self._n}",
        stroke="white", fill="none",
        stroke_width=max(1, self.size * 0.09),
        stroke_linecap="round", opacity=0.55)

@patch
def register_styles(self: DiagramGlyphs, builder: SVGBuilder):
    """Register all glyph CSS classes on an SVGBuilder."""
    for fn in (self.outline_style, self.filled_style, self.accent_style,
               self.solid_style, self.stalk_style, self.band_style,
               self.danger_style, self.blocked_fill_style, self.blocked_cross_style):
        builder.add_style(fn())


@patch
def blocked_x(self: DiagramGlyphs, center: MapCord, r=None) -> str:
    """Red circle with white X cross — blocked/impassable move marker."""
    r = r or self.size * 0.25
    svg = self.circle(center, r, self.blocked_fill_style())
    d = r * 0.55
    for dx1, dy1, dx2, dy2 in [(-d, -d, d, d), (d, -d, -d, d)]:
        svg += MapPath([
            MapCord(center.x + dx1, center.y + dy1),
            MapCord(center.x + dx2, center.y + dy2),
        ], self.blocked_cross_style()).drawPolygon()
    return svg


@patch
def start_marker(self: DiagramGlyphs, center: MapCord, r=None,
                 style: StyleCSS = None) -> str:
    """Hollow ring drawn around the piece's start position."""
    r = r or self.size * 0.8
    return self.ring(center, r, style or self.outline_style())


@patch
def step_label(self: DiagramGlyphs, center: MapCord, n: int,
               offset=None, font_size=None, color=None) -> str:
    """Small step-number badge offset up-right from a hex centre.

    Args:
        n: step number (0-indexed internally; displayed as n+1)
        offset: pixel offset from centre (defaults to size * 0.35)
        font_size: override font size (defaults to max(12, size * 0.3))
        color: text fill (defaults to flag accent/comp color)
    """
    offset    = offset    or self.size * 0.35
    font_size = font_size or max(12, self.size * 0.3)
    color     = color     or self.accent
    return (f'<text x="{center.x + offset:.1f}" y="{center.y - offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{n+1}</text>\n')


@patch
def action_glyph(self: DiagramGlyphs, instr: 'Instruction',
                 center: MapCord, style: StyleCSS = None) -> str:
    """Dispatch a non-movement instruction to its geometry glyph.

    Returns empty string for movement instructions (FORWARD) — those
    are handled separately as arrows.
    """
    style = style or self.filled_style()

    dispatch = {
        Instruction.PAUSE:   lambda: self.pause(center, style=style),
        Instruction.DEFEND:  lambda: self.shield(center, style=style),
        Instruction.HARVEST: lambda: self.sheaf(center),
        Instruction.GIVE:    lambda: self.gift(center, style=style),
        Instruction.SETTLE:  lambda: self.house(center, style=style),
        Instruction.ROT_L:   lambda: self.rot_arrow(center, clockwise=False),
        Instruction.ROT_R:   lambda: self.rot_arrow(center, clockwise=True),
    }
    fn = dispatch.get(instr)
    return fn() if fn else ""


In [ ]:
#| export
@patch
def bar_style(self: DiagramGlyphs) -> StyleCSS:
    """Thick round-cap stroke for facing bars."""
    return StyleCSS(f"glyph_bar_{self._n}",
        stroke=self.color, fill="none",
        stroke_width=max(2, self.size * 0.15),
        stroke_linecap="round", opacity=self.opacity)

@patch
def register_styles(self: DiagramGlyphs, builder: SVGBuilder):
    """Register all glyph CSS classes on an SVGBuilder."""
    for fn in (self.outline_style, self.filled_style, self.accent_style,
               self.solid_style, self.stalk_style, self.band_style,
               self.danger_style, self.blocked_fill_style, self.blocked_cross_style,
               self.bar_style):
        builder.add_style(fn())


@patch
def facing_bar(self: DiagramGlyphs, center: MapCord, facing: int,
               length=None, offset=None,
               style: StyleCSS = None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    length = length or self.size * 1.2
    offset = offset or self.size * 1.1
    style  = style  or self.bar_style()

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = center.x + offset * math.cos(angle)
    by = center.y + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    p1 = MapCord(bx + half * math.cos(perp), by + half * math.sin(perp))
    p2 = MapCord(bx - half * math.cos(perp), by - half * math.sin(perp))

    return MapPath([p1, p2], style).drawPolygon()


I am refactoring DiagramGlyphs so that it uses StyleCSS, CountryFlag, and existing primitives like MapCord and MapPath as much as possible. You are doing a great job. can you rewrite facing_bar using them?

Can you build a nice gallery demo of DiagramGlyphs("Death ✕",      lambda c: glyphs.death_marker(c)),

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
glyphs = DiagramGlyphs(flag, size=22)

canvas = SVGBuilder()
canvas.add_font('Cinzel')
glyphs.register_styles(canvas)

# Also register the dynamic pip styles we'll use
for pct in [0, 25, 50, 75, 100]:
    f = pct / 100
    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
    canvas.add_style(StyleCSS(f"pip_{pct}", fill=color, stroke=color, stroke_width=0.8, opacity=0.85))

cols, rows = 5, 4
cell_w, cell_h = 110, 100
margin = 20
canvas.width = cols * cell_w + margin * 2
canvas.height = rows * cell_h + margin * 2 + 30

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="{margin + 6}" text-anchor="middle" '
    f'font-size="15" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'DiagramGlyphs Gallery — {flag.name}\'s {flag.countryPrefix}</text>')

# Glyph catalog: (name, draw_fn)
catalog = [
    ("Circle",       lambda c: glyphs.circle(c, 18)),
    ("Ring",         lambda c: glyphs.ring(c, 20)),
    ("Arc",          lambda c: glyphs.arc(c, 18, -30, 210)),
    ("Shield",       lambda c: glyphs.shield(c)),
    ("Sheaf",        lambda c: glyphs.sheaf(c)),
    ("Gift",         lambda c: glyphs.gift(c)),
    ("House",        lambda c: glyphs.house(c)),
    ("Facing Bar",   lambda c: glyphs.facing_bar(c, 0)),
    ("Pause",        lambda c: glyphs.pause(c)),
    ("Rot → CW",     lambda c: glyphs.rot_arrow(c, clockwise=True)),
    ("Rot ← CCW",    lambda c: glyphs.rot_arrow(c, clockwise=False)),
    ("Death ✕",      lambda c: glyphs.death_marker(c)),
    ("Blocked ⊘",    lambda c: glyphs.blocked_x(c)),
    ("Start Ring",   lambda c: glyphs.start_marker(c)),
    ("Step Label",   lambda c: glyphs.step_label(c, 4)),
    ("Transfer →",   lambda c: glyphs.transfer_arrow(
                         MapCord(c.x - 30, c.y), MapCord(c.x + 30, c.y))),
    # Food pips at various hunger levels
    ("Pip 0%",       lambda c: glyphs.food_pip(c, 0.0)),
    ("Pip 25%",      lambda c: glyphs.food_pip(c, 0.25)),
    ("Pip 50%",      lambda c: glyphs.food_pip(c, 0.50)),
    ("Pip 75%",      lambda c: glyphs.food_pip(c, 0.75)),
    ("Pip 100%",     lambda c: glyphs.food_pip(c, 1.0)),
]

body = ''
lbl_style = flag.labelStyle("gallery_lbl")
canvas.add_style(lbl_style)

for i, (name, draw_fn) in enumerate(catalog):
    row, col = divmod(i, cols)
    cx = margin + col * cell_w + cell_w // 2
    cy = margin + 30 + row * cell_h + cell_h // 2 - 8

    # Light background disc for each cell
    body += (f'<circle cx="{cx}" cy="{cy}" r="32" '
             f'fill="{flag.lightPrimary}" opacity="0.35"/>\n')

    # Draw the glyph
    body += draw_fn(MapCord(cx, cy))

    # Label below
    body += (f'<text x="{cx}" y="{cy + 44}" text-anchor="middle" '
             f'font-size="9" font-family="\'Cinzel\', sans-serif" '
             f'class="{lbl_style.name}">{name}</text>\n')

canvas.adjust("glyphs", body)
canvas.show()


HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_SAADm9NCSzO0nm4AVa4rpw&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css&quot;&gt;
    &lt;style&gt;:root { --pico-font-size: 100%; }&lt;/style&gt;
    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;    &lt;link href=&quot;https://cdn.jsdelivr.net/npm/daisyui@5/themes.css&quot; rel=&quot;stylesheet&quot; type=&quot;text/css&quot;&gt;
&lt;script&gt;
    function sendmsg() {
        window.parent.postMessage({height: document.documentElement.offsetHeight}, &#x27;*&#x27;);
    }
    window.onload = function() {
        sendmsg();
        document.body.addEventListener(&#x27;htmx:afterSettle&#x27;,    sendmsg);
        document.body.addEventListener(&#x27;htmx:wsAfterMessage&#x27;, sendmsg);
    };&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div&gt;&lt;?xml version=&#x27;1.0&#x27; encoding=&#x27;utf-8&#x27;?&gt;
&lt;svg  width=&quot;590&quot; height=&quot;470&quot; viewBox=&quot;0 0 590 470&quot; xmlns=&quot;http://www.w3.org/2000/svg&quot;&gt;
&lt;title&gt; Untitled &lt;/title&gt;
  &lt;style&gt;
@import url(&#x27;https://fonts.googleapis.com/css2?family=Cinzel&amp;display=swap&#x27;);
.pip_75 {
  fill:#43bc2c;
  stroke:#43bc2c;
  stroke-width:0.8;
  opacity:0.85;
}
.pip_50 {
  fill:#a3cb31;
  stroke:#a3cb31;
  stroke-width:0.8;
  opacity:0.85;
}
.pip_25 {
  fill:#d9a436;
  stroke:#d9a436;
  stroke-width:0.8;
  opacity:0.85;
}
.pip_100 {
  fill:#27ae60;
  stroke:#27ae60;
  stroke-width:0.8;
  opacity:0.85;
}
.pip_0 {
  fill:#e74c3c;
  stroke:#e74c3c;
  stroke-width:0.8;
  opacity:0.85;
}
.glyph_stalk_Harold {
  stroke:#562730;
  fill:none;
  stroke-width:3.3;
  stroke-linecap:round;
  opacity:0.85;
}
.glyph_solid_Harold {
  stroke:none;
  fill:#562730;
  opacity:0.85;
}
.glyph_fill_Harold {
  stroke:#562730;
  fill:#ffc8d1;
  stroke-width:1.5;
  opacity:0.85;
}
.glyph_danger_Harold {
  stroke:#c0392b;
  fill:none;
  stroke-width:3.0;
  stroke-linecap:round;
  opacity:0.85;
}
.glyph_blocked_x_Harold {
  stroke:white;
  fill:none;
  stroke-width:1.98;
  stroke-linecap:round;
  opacity:0.55;
}
.glyph_blocked_Harold {
  fill:red;
  stroke:darkred;
  stroke-width:1;
  opacity:0.55;
}
.glyph_bar_Harold {
  stroke:#562730;
  fill:none;
  stroke-width:3.3;
  stroke-linecap:round;
  opacity:0.85;
}
.glyph_band_Harold {
  stroke:#562730;
  fill:none;
  stroke-width:2.2;
  opacity:0.85;
}
.glyph_accent_Harold {
  stroke:#27564e;
  fill:#27564e;
  opacity:0.85;
}
.glyph_Harold {
  stroke:#562730;
  fill:none;
  stroke-width:1.5;
  opacity:0.85;
  stroke-linecap:round;
}
.contrast_gallery_lbl {
  fill:#562730;
  stroke:none;
  stroke-width:0;
}
  &lt;/style&gt;

&lt;g data-layer=&quot;root&quot;&gt;

&lt;/g&gt;
&lt;g data-layer=&quot;title&quot;&gt;
&lt;text x=&quot;295&quot; y=&quot;26&quot; text-anchor=&quot;middle&quot; font-size=&quot;15&quot; font-family=&quot;&#x27;Cinzel&#x27;, sans-serif&quot; fill=&quot;#562730&quot;&gt;DiagramGlyphs Gallery — Harold&#x27;s Habitat&lt;/text&gt;
&lt;/g&gt;
&lt;g data-layer=&quo

## context

In [ ]:
#| export
@dataclass
class GameContext(OverlayContext):
    """Rendering context with game-layer data (board, corridors, pieces)."""

    @property
    def board(self): return self.extras.get('board')

    @property
    def corridors(self): return self.extras.get('corridors', [])

    @property
    def pieces(self): return self.extras.get('pieces', [])

    @property
    def squads(self): return self.extras.get('squads', [])

    @property
    def simulator(self):
        return self.extras.get('simulator')

   


## Style guide

So when I prompt for things in other notebooks it doesn't do DiagramGlyphs, StyleCSS/SVGBuilder, or any of our primtives see
```

    if turns_left >= reserve:
        bar_color = "#27ae60"
    elif turns_left >= reserve * 0.5:
        bar_color = "#f39c12"
    else:
        bar_color = "#e74c3c"

    gauge_svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" '
        f'viewBox="0 0 {w} {h}">'
        # Background track
        f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" fill="#eee"/>'
        # Fill bar
        f'<rect x="0" y="{bar_y}" width="{w * frac:.1f}" height="{bar_h}" '
        f'rx="3" fill="{bar_color}" opacity="0.8"/>'
        # Reserve threshold tick
        f'<line x1="{w * reserve_frac:.1f}" y1="{bar_y - 1}" '
        f'x2="{w * reserve_frac:.1f}" y2="{bar_y + bar_h + 1}" '
        f'stroke="#e67e22" stroke-width="2" stroke-dasharray="2,1"/>'
        # Labels
        f'<text x="4" y="{bar_y + bar_h + 14}" font-size="9" fill="#555" '
        f'font-family="sans-serif">'
        f'🌾 {turns_left:.1f} turns left</text>'
        f'<text x="{w - 4}" y="{bar_y + bar_h + 14}" font-size="9" fill="#e67e22" '
        f'font-family="sans-serif" text-anchor="end">'
        f'reserve={reserve}</text>'
        f'</svg>'
    )
```
What I need is a style guilde that tells them how to build things in existing codebase? can you write something?

# HexMagic SVG Style Guide

**Never write raw inline SVG.** Use the existing primitives below.

## Core Primitives

### Coordinates & Paths
```python
from HexMagic.primitives import MapCord, MapPath

pt = MapCord(100, 50)                      # a point
path = MapPath([pt1, pt2, pt3], style)     # styled polyline
path.drawClosed()                          # closed polygon SVG string
path.drawPolygon()                         # open polyline SVG string
path.with_arrowhead(arrow_size=6)          # line with arrowhead
path.make_windy(iterations=3, offset_factor=0.12)  # wavy edge
```

### SVGBuilder — the canvas
```python
from HexMagic.styles import SVGBuilder, StyleCSS

canvas = SVGBuilder()
canvas.width = 400; canvas.height = 300
canvas.add_font('Cinzel')                  # Google font
canvas.add_style(style)                    # register a StyleCSS
canvas.add_definition(pat)                 # register an SVGDef (pattern, clipPath…)
canvas.adjust("layer_name", svg_string)    # add/replace a named layer
canvas.show()                              # display in notebook
canvas.xml()                               # raw SVG string (for FastHTML)
```

### StyleCSS — named CSS classes
```python
style = StyleCSS("my_bar",
    fill="#27ae60", stroke="#333",
    stroke_width=2, opacity=0.8,
    stroke_linecap="round")

# Use via class= in SVG elements:
f'<rect ... class="{style.name}"/>'

# Color utilities:
StyleCSS.lerp_color('#e74c3c', '#27ae60', 0.5)  # interpolate colors
```

**Do this:**
```python
style = StyleCSS("food_bar", fill=bar_color, stroke="none", opacity=0.8)
canvas.add_style(style)
f'<rect x="0" y="{y}" width="{w}" height="{h}" rx="3" class="{style.name}"/>'
```

**Not this:**
```python
f'<rect ... fill="{bar_color}" opacity="0.8"/>'  # ❌ inline styles
```

## CountryFlag — color theming

A `CountryFlag` holds a harmonized color palette derived from one base color:

| Property       | Use for                          |
|----------------|----------------------------------|
| `flag.primary` | backgrounds, piece body fill     |
| `flag.comp`    | text, strokes, contrast elements |
| `flag.darkPrimary` | labels, dark accents        |
| `flag.lightPrimary` | subtle backgrounds         |
| `flag.lightComp` | icon circle backgrounds        |
| `flag.tri1`, `flag.tri2` | triadic accent colors |

### Named style factories
```python
flag.plain("name")          # fill=primary, stroke=comp
flag.labelStyle("name")     # fill=darkPrimary, no stroke (for text)
flag.contrastStyle("name")  # fill=comp, stroke=black
flag.kingStyle("name")      # semi-transparent primary
```

### Rendering pieces (chess glyphs)
```python
from HexMagic.game.flag import PieceType

# Full pipeline — registers pattern on builder automatically:
flag.draw_piece(PieceType.QUEEN, MapCord(cx, cy), canvas,
                scale=1.5, size='board', layer="my_queen")

# size tiers:
#   'board'  → solid flag.primary fill (tiny, ~0.5 scale)
#   'list'   → simple geometric pattern (medium, ~1.5 scale)
#   'large'  → full decorative pattern (big, ~3.5 scale)
```

### Rendering flags
```python
flag.draw_flag(MapCord(cx, cy), canvas,
               scale=2.0, size='list', wavy=True, pole=True,
               layer="my_flag")
```

### Rendering animal icons
```python
# Deterministic animal per flag (stable across calls):
flag.animal_svg(size='list', center=MapCord(cx, cy), piece_id="icon_1")

# Specific animal:
flag.draw_icon(svg_source, MapCord(cx, cy), canvas,
               scale=1.5, piece_id="bear_1", layer="bear")

# Squad names:
for animal, squad in flag.country_squads(6):
    print(squad)  # e.g. "Fierce Foxes"
```

## DiagramGlyphs — overlay symbols

For UI overlays, gauges, markers — **not raw SVG shapes**.

```python
glyphs = DiagramGlyphs(flag, size=22)
glyphs.register_styles(canvas)  # registers all CSS classes at once

# Markers & shapes
glyphs.circle(center, r=18)
glyphs.ring(center, r=20)
glyphs.arc(center, r=18, start_deg=-30, end_deg=210)
glyphs.shield(center)
glyphs.house(center)
glyphs.sheaf(center)            # wheat/harvest
glyphs.gift(center)
glyphs.pause(center)

# Movement & facing
glyphs.rot_arrow(center, clockwise=True)
glyphs.facing_bar(center, facing=0)
glyphs.transfer_arrow(start, end)

# Status indicators
glyphs.food_pip(center, frac=0.75)    # 0→red, 1→green, sized by frac
glyphs.death_marker(center)            # bold red X
glyphs.blocked_x(center)              # red circle + white X
glyphs.start_marker(center)           # hollow ring
glyphs.step_label(center, n=3)        # numbered badge
```

### Example: food gauge (the right way)

```python
glyphs = DiagramGlyphs(flag, size=14)
glyphs.register_styles(canvas)

# Background track
track = StyleCSS("track", fill="#eee", stroke="none")
canvas.add_style(track)
canvas.adjust("track", f'<rect x="0" y="{y}" width="{w}" height="{h}" rx="3" class="{track.name}"/>')

# Fill bar — use lerp_color for smooth red→green
bar_color = StyleCSS.lerp_color('#e74c3c', '#27ae60', frac)
bar = StyleCSS("food_fill", fill=bar_color, stroke="none", opacity=0.8)
canvas.add_style(bar)
canvas.adjust("fill", f'<rect x="0" y="{y}" width="{w * frac:.1f}" height="{h}" rx="3" class="{bar.name}"/>')

# Threshold tick
tick = glyphs.outline_style()
canvas.adjust("tick", MapPath([
    MapCord(w * reserve_frac, y - 1),
    MapCord(w * reserve_frac, y + h + 1)
], tick).drawPolygon())

# Labels — use flag.labelStyle, not inline fill=
lbl = flag.labelStyle("gauge_lbl")
canvas.add_style(lbl)
canvas.adjust("label", f'<text x="4" y="{ty}" font-size="9" class="{lbl.name}">🌾 {turns_left:.1f}</text>')
```

## Anti-patterns

| ❌ Don't | ✅ Do |
|----------|-------|
| `fill="#27ae60"` inline | `StyleCSS("name", fill=...)` + `class=` |
| `<svg xmlns=...>` wrapper per widget | `SVGBuilder` canvas |
| Hardcoded color literals | `flag.primary`, `flag.comp`, `StyleCSS.lerp_color(...)` |
| Manual `<circle>`, `<path>` for markers | `glyphs.food_pip()`, `glyphs.shield()`, etc. |
| Raw SVG polygon strings | `MapPath([...], style).drawClosed()` |
| `font-family="sans-serif"` everywhere | `canvas.add_font('Cinzel')` + `flag.labelStyle()` |
| Separate `<svg>` per small widget | Compose layers on one `SVGBuilder` |
